## Juan Camilo Bedoya y Linda Catalina Correa

In [1]:
# ============================================================
# Celda 0: imports y configuración general
# ============================================================
import os, io, zipfile, re, math, gc
from collections import defaultdict
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["font.size"] = 10

DATA_DIR = "./data"
os.makedirs(DATA_DIR, exist_ok=True)

IGRA_LIST_LOCAL   = "./data/IGRA/igra2-station-list.txt"
IGRA_FORMAT_LOCAL = "./data/IGRA/igra2-data-format.txt"  # referencia documental (no se usa para parsing)
os.makedirs(os.path.dirname(IGRA_LIST_LOCAL), exist_ok=True)


In [2]:
# ============================================================
# Celda 1: lectura robusta de la lista de estaciones (anchos fijos)
# Objetivo en el flujo del trabajo:
# - Cargar el inventario oficial de estaciones IGRA v2 desde NOAA.
# - Parsear el archivo de "anchos fijos" (fixed-width) con pandas.
# - Tipar columnas numéricas y filtrar estaciones inválidas/móviles.
# ============================================================

# Descarga el listado si no existe localmente
if not os.path.exists(IGRA_LIST_LOCAL):
    # URL del listado oficial de estaciones IGRA v2 (documentación de NCEI/NOAA)
    url_list = "https://www.ncei.noaa.gov/data/integrated-global-radiosonde-archive/doc/igra2-station-list.txt"
    r = requests.get(url_list, timeout=60)
    r.raise_for_status()
    # Guarda el contenido en disco como copia local reutilizable
    with open(IGRA_LIST_LOCAL, "wb") as f:
        f.write(r.content)

# Colspecs según documento IGRA v2 (1-index en doc → aquí 0-index [start, end))
# Definimos los rangos de columnas (inicio, fin) para cada campo del archivo de anchos fijos.
colspecs = [
    (0, 11),   # ID             → identificador IGRA de la estación (11 chars)
    (12, 20),  # LATITUDE       → latitud en grados decimales (texto en archivo)
    (21, 30),  # LONGITUDE      → longitud en grados decimales (texto en archivo)
    (31, 37),  # ELEVATION      → elevación en metros
    (38, 40),  # STATE          → código de estado/provincia (si aplica; sobre todo para EE. UU.)
    (41, 71),  # NAME           → nombre de la estación (puede traer espacios)
    (72, 76),  # FSTYEAR        → primer año con observaciones
    (77, 81),  # LSTYEAR        → último año con observaciones
    (82, 88),  # NOBS           → número total de sondeos (periodo completo)
]
# Nombres de las columnas que pondrá pandas al leer el archivo
names = ["id","lat","lon","elev_m","state","name","fstyear","lstyear","nobs"]

# Lee el archivo de anchos fijos como texto crudo primero (dtype=str) para evitar
# que pandas rompa filas raras o haga conversiones prematuras. Luego tipamos nosotros.
stations = pd.read_fwf(IGRA_LIST_LOCAL, colspecs=colspecs, names=names, dtype=str)

# Tipos numéricos y limpieza
# Convierte lat, lon y elevación a float (coerce: valores inválidos → NaN en vez de romper)
for c in ["lat","lon","elev_m"]:
    stations[c] = pd.to_numeric(stations[c], errors="coerce")
# Convierte años (fstyear, lstyear) y recuento (nobs) a enteros "Int64" (permite NaN)
for c in ["fstyear","lstyear","nobs"]:
    stations[c] = pd.to_numeric(stations[c], errors="coerce").astype("Int64")

# Quita estaciones móviles/sin coord válidas
# IGRA usa sentinelas: lat = -98.8888 o lon = -998.8888 para marcar coordenadas inválidas/móviles.
# Los filtramos para no considerar estaciones que no se pueden ubicar espacialmente.
stations["id"] = stations["id"].str.strip()
stations = stations[
    stations[["id", "lat", "lon"]].notna().all(axis=1) &
    (~np.isclose(stations["lat"], -98.8888, equal_nan=True)) &
    (~np.isclose(stations["lon"], -998.8888, equal_nan=True))
].drop_duplicates("id", keep="last").copy()

# Muestra una vista rápida (3 primeras filas) para verificar que el parseo fue correcto
stations.head(3)


,id,lat,lon,elev_m,state,name,fstyear,lstyear,nobs
0,ACM00078861,17.1170,-61.783,10.0,NaN,COOLIDGE FIELD (UA),1947,1993,13896
1,AEM00041217,24.4333,54.650,16.0,NaN,ABU DHABI INTERNATIONAL AIRPOR,1983,2025,40887
2,AEXUAE05467,25.2500,55.370,4.0,NaN,SHARJAH,1935,1942,2477


In [3]:
# ============================================================
# Celda 2: filtro espacial (caja MJO) y candidatos por NOBS
# - Seleccionar las estaciones IGRA ubicadas dentro de una "caja MJO" tropical,
#   donde el MJO tiene mayor relevancia (trópicos del Índico → Pacífico).
# - Ordenar/inspeccionar por NOBS (número total de sondeos) para priorizar
#   estaciones con registros largos/continuos en los siguientes análisis.
#   Estas candidatas alimentarán la descarga, el parseo y los composites RMM.
# ============================================================

LON_MIN, LON_MAX = 65.0, 160.0
LAT_MIN, LAT_MAX = -10.0, 10.0

# Aplica el filtro espacial sobre el DataFrame 'stations' leído en la Celda 1:
# - Mantén solo filas cuya longitud (lon) esté entre LON_MIN y LON_MAX
#   y cuya latitud (lat) esté entre LAT_MIN y LAT_MAX.
# - .copy() crea una copia independiente para evitar advertencias por "view" de pandas
#   cuando modifiques este subconjunto más adelante.
mjo_stations = stations[
    (stations["lon"] >= LON_MIN) & (stations["lon"] <= LON_MAX) &
    (stations["lat"] >= LAT_MIN) & (stations["lat"] <= LAT_MAX)
].copy()

print(f"Estaciones en caja MJO [{LON_MIN}–{LON_MAX}E, {LAT_MIN}–{LAT_MAX}°]: {len(mjo_stations)}")

# Ordena esas estaciones por 'nobs' (número total de sondeos) de mayor a menor
# y muestra las 10 primeras. Esto sirve para:
# - Evaluar cuáles tienen mejor cobertura temporal y densidad de observaciones,
# - Elegir 2 estaciones “ganadoras” para el análisis (descarga, θ, N², dθ/dz, composites RMM).
mjo_stations.sort_values("nobs", ascending=False).head(10)


Estaciones en caja MJO [65.0–160.0E, -10.0–10.0°]: 147


,id,lat,lon,elev_m,state,name,fstyear,lstyear,nobs
1956,SNM00048698,1.3679,103.9824,14.0,NaN,SINGAPORE/CHANGI AIRPORT,1955,2025,72262
2034,THM00048568,7.1819,100.6075,4.6,NaN,SONGKHLA,1954,2025,70804
1490,MYM00048601,5.3000,100.2667,2.0,NaN,PENANG/BAYAN LEPAS,1968,2025,65891
1492,MYM00048615,6.1667,102.3006,4.4,NaN,KOTA BHARU,1973,2025,61237
1498,MYM00096413,1.4903,110.3525,20.9,NaN,KUCHING,1973,2025,60636
1499,MYM00096441,3.1200,113.0246,23.0,NaN,BINTULU,1973,2025,59712
1127,INM00043371,8.4833,76.9500,59.7,NaN,THIRUVANANTHAPURAM (43371-0),1928,2025,59692
1500,MYM00096471,5.9333,116.0500,2.1,NaN,KOTA KINABALU,1969,2025,59024
791,FMM00091348,6.9500,158.2000,38.0,NaN,PONAPE/CAROLINE IS.,1950,2025,56222
1497,MYM00048657,3.7722,103.2119,15.2,NaN,KUANTAN,1971,2025,55942


In [4]:
# ============================================================
# Celda 3: utilidades de parseo IGRA
# - Convertir el archivo de sondeos IGRA (líneas de texto de anchos fijos) a
#   estructuras Python (dict/DataFrame) con columnas físicas (p, T, RH, etc.).
# - Respetar el formato v2 oficial: posiciones exactas por campo.
# - Normalizar "missing/QC" (-8888, -9999) a NaN para cálculos posteriores.
# ============================================================

QC_REMOVE = {-8888, -9999}  # códigos de missing/QC IGRA (aparecen en muchas variables)
#  Conjunto de sentinelas que IGRA usa para "dato eliminado por QC" o "faltante".
#   Los convertiremos a NaN para que pandas/numpy los ignoren en medias, percentiles, etc.

QC_LETTER_RE = re.compile(r"[A-Za-z]")
#  Expresión regular para quitar letras que a veces vienen "pegadas" a los números
#   (flags de control de calidad). Así limpiamos '9999A' → '9999' antes de parsear.

def _to_int_field(token: str):
    """Convierte campo con posibles flags (letras) a int o None."""
    #  Helper robusto para leer segmentos crudos (texto) como enteros, tolerando flags.
    if token is None: 
        return None
    t = QC_LETTER_RE.sub("", token.strip())
    #  Elimina espacios y cualquier letra (flags), dejando solo signos y dígitos.
    if t == "": 
        return None
    try:
        return int(t)
    except:
        return None
    #  Si no se puede convertir a int (caracteres raros), devolvemos None (se tratará como NaN después).

def parse_header_line(h: str):
    """Header IGRA (línea que comienza con '#'). Extrae ID y timestamp UTC."""
    #  Cada sondeo empieza con un header '#...' que contiene metadatos del sondeo.
    if not h.startswith("#"):
        raise ValueError("Header no comienza con '#'.")
    #  Validación básica: un header válido debe empezar con '#'.

    sid   = h[1:12].strip()
    #  ID de estación IGRA. El formato IGRA usa columnas fijas (1-index en doc).
    #   Aquí cortamos con índices 0-index de Python [1:12).

    year  = _to_int_field(h[13:17])
    month = _to_int_field(h[18:20])
    day   = _to_int_field(h[21:23])
    hour  = _to_int_field(h[24:26])
    #  Año/mes/día/hora del sondeo. _to_int_field limpia e intenta convertir a int.

    ts = pd.Timestamp(datetime(int(year), int(month), int(day), 0 if hour in (None, 99) else int(hour)))
    #  Construye el timestamp UTC del lanzamiento.
    #   Nota: algunos registros traen HOUR=99 (desconocido); en ese caso fijamos 00 UTC
    #   para no romper el time-index. Esto permite agrupar por fecha y empatar con RMM.

    numlev = _to_int_field(h[32:36])  # aprox niveles
    #  NUMLEV: contador aproximado de niveles reportados en el sondeo (útil para diagnóstico).

    return {"station_id": sid, "time": ts, "numlev": numlev}
    #  Devolvemos un dict mínimo con lo que necesitamos para catalogar y etiquetar el sondeo.

def parse_level_line(ln: str) -> dict:
    """Registro de nivel IGRA (formato de columnas fijas)."""
    #  Cada línea (no-#) del archivo representa un "nivel" (superficie, estándar, significativos, etc.)
    #   con variables meteorológicas medidas/derivadas.
    def gi(a,b): return ln[a-1:b]  # helper 1-index → slice python
    #  Helper que convierte los offsets 1-index de la documentación IGRA
    #   a cortes Python (0-index). gi(10,15) extrae [9:15).
    def tint(a,b): return _to_int_field(gi(a,b))
    #  Helper que además de cortar, intenta convertir el segmento a int (limpiando flags).

    # Campos que necesitamos
    p_pa   = tint(10,15)      # presión (Pa o mb*100)
    z_m    = tint(17,21)      # altura geopotencial (m)
    t_t    = tint(23,27)      # temperatura (0.1 °C)
    rh_t   = tint(29,33)      # Humedad relativa (0.1 %)
    dpd_t  = tint(35,39)      # depresión de rocío (0.1 °C)
    wdir   = tint(41,45)      # viento dir (°)
    wspd_t = tint(47,51)      # viento vel (0.1 m/s)
    #  Tomamos solo las variables necesarias para el análisis:
    #   - p, z, T, RH/DPD (para θ, θe, N², dθ/dz),
    #   - viento (por si evaluamos dinámicas o filtros de calidad).
    #   Otras columnas IGRA existen, pero no son imprescindibles aquí.

    # Conversión simple + NaN en códigos QC
    def from_tenths(v, scale=10.0):
        if v is None or v in QC_REMOVE: return np.nan
        return v/scale
    #  Convierte números almacenados en décimas (ej. 235 → 23.5) a unidades reales.
    #   Si el valor es None o un sentinela QC, devolvemos NaN.

    def p_to_hPa(v):
        if v is None or v in QC_REMOVE: return np.nan
        return v/100.0
    #  IGRA guarda presión en Pa o "mb*100". Dividir entre 100 → hPa.
    #   Esto normaliza unidades para perfiles T–p y cálculo de θ.

    def wspd_mps(v):
        if v is None or v in QC_REMOVE: return np.nan
        return v/10.0
    #  Velocidad del viento viene en décimas de m/s; la llevamos a m/s.

    return {
        "pressure_hPa": p_to_hPa(p_pa),
        #  Presión en hPa (base de casi todo: orden del perfil, θ, hipsométrica).

        "height_m":     np.nan if (z_m is None or z_m in QC_REMOVE) else float(z_m),
        #  Altura geopotencial (m). Si falta, más adelante se puede estimar con hipsométrica.

        "temp_C":       from_tenths(t_t),
        #  Temperatura ambiente (°C), imprescindible para θ, θe y lapsos.

        "rh_pct":       from_tenths(rh_t),
        #  Humedad relativa (%) si está disponible. Útil para estimar Td y θe cuando no hay DPD.

        "dpd_C":        from_tenths(dpd_t),
        #  Depresión de punto de rocío (°C). Con T y DPD puedes obtener Td = T - DPD.

        "wind_dir_deg": np.nan if (wdir is None or wdir in QC_REMOVE) else float(wdir),
        #  Dirección del viento (grados). No entra directo en N² o dθ/dz

        "wind_speed_mps": wspd_mps(wspd_t),
        # ^ Velocidad del viento (m/s).

    }
    #  Este dict por nivel alimentará un DataFrame de perfil por sondeo.
    #   Luego calcularemos θ, Δz hipsométrico, dθ/dz(0–3 km) y N²(980–850 hPa),
    #   que son las métricas clave de nuetsro objetivo (preacondicionamiento MJO).


In [5]:
# ============================================================
# Celda 4: partir archivo en sondeos y construir catálogo + meta
# - Un archivo IGRA por estación contiene MUCHOS sondeos (cada uno inicia con '#').
# - Aquí los separamos en bloques (header + niveles) y convertimos cada sondeo a DataFrame.
# - Además, creamos un "catálogo" (dict clave→perfil) y una tabla "meta" con estadísticas por sondeo.
# - Esto alimenta: cálculos de θ, dθ/dz, N², y el emparejamiento con RMM por fecha.
# ============================================================

def iter_soundings(lines):
    """Itera (header_line, [level_lines...]) sin cargar el archivo completo en RAM."""
    cur_h, cur_blk = None, []
    for ln in lines:                      # recorre línea por línea el archivo de la estación
        if not ln.strip():                # ignora líneas vacías (solo espacios/saltos)
            continue
        if ln.startswith("#"):            # si la línea comienza con '#', es un NUEVO header de sondeo
            if cur_h is not None:         # si ya teníamos un sondeo abierto, lo cerramos y guardamos
                yield cur_h, cur_blk
            cur_h, cur_blk = ln, []       # iniciamos un nuevo sondeo: header actual y bloque vacío
        else:
            cur_blk.append(ln)            # si no es '#', es línea de nivel → la añadimos al bloque del sondeo
    if cur_h is not None:                 # al terminar el loop, si hay un sondeo abierto, guardarlo
        yield cur_h, cur_blk

def split_soundings(lines):
    """Compatibilidad: materializa la lista solo cuando se solicita explícitamente."""
    return list(iter_soundings(lines))

def build_catalog(lines, drop_empty=True):
    """cat: dict[str->DataFrame niveles], meta: DataFrame por sondeo."""
    cat, meta = {}, []                    # cat: clave→DataFrame del perfil; meta: filas con info/estadísticas por sondeo
    dup_count = defaultdict(int)          # para manejar sondeos con timestamps duplicados: lleva un contador por fecha

    for h, block in iter_soundings(lines):          # itera cada sondeo sin duplicar todo el archivo en memoria
        hd = parse_header_line(h)                   # parsea header → {'station_id','time','numlev'}
        rows = [parse_level_line(ln) for ln in block]  # parsea cada línea de nivel a dict con variables físicas
        df = pd.DataFrame(rows)                     # DataFrame del perfil en ese sondeo (columnas: p, z, T, RH, etc.)
        df.insert(0, "station", hd["station_id"])   # inserta ID de estación (útil para agrupar/filtrar después)
        df.insert(1, "time", hd["time"])            # inserta timestamp UTC del sondeo (para empatar con RMM)

        # Conteos de validez por variable clave: sirven para clasificar el tipo de sondeo
        nP = df["pressure_hPa"].notna().sum()       # cuántos niveles tienen presión válida
        nT = df["temp_C"].notna().sum()             # cuántos niveles tienen temperatura válida
        nW = df["wind_speed_mps"].notna().sum()     # cuántos niveles tienen viento válido

        # Clasificación básica del sondeo según disponibilidad de variables
        if nP==0 and nT==0 and nW==0: status="empty"          # completamente vacío (no utilizable)
        elif nP==0 and nT==0 and nW>0: status="winds_only"    # solo viento (sin p/T → no sirve para θ/N²)
        elif nT==0 and nP>0:          status="pressure_only"  # hay presión, pero falta T → térmico incompleto
        else:                         status="full"           # tiene presión y T (lo necesitas para θ, dθ/dz, N²)

        # Ordena por presión (descendente: superficie→tope) o por altura (ascendente) si no hay presión
        if df["pressure_hPa"].notna().any():                 # si hay presión en algún nivel:
            df = df.sort_values("pressure_hPa", ascending=False).reset_index(drop=True)
        elif df["height_m"].notna().any():                   # si no hay p pero hay z, ordena de abajo hacia arriba
            df = df.sort_values("height_m", ascending=True).reset_index(drop=True)

        # clave única (desambiguando duplicados con sufijo .2, .3...)
        ts = hd["time"]                                      # timestamp del sondeo (puede repetirse en el archivo)
        dup_count[ts] += 1                                   # incrementa contador para esa fecha/hora
        key = ts.isoformat() + ("" if dup_count[ts]==1 else f".{dup_count[ts]}")
        cat[key] = df                                        # guarda el perfil en el catálogo bajo esa clave

        # meta simple: estadísticas rápidas por sondeo (útil para exploración y filtros posteriores)
        pmax = float(df["pressure_hPa"].max()) if df["pressure_hPa"].notna().any() else np.nan  # pres máx ~ superficie
        pmin = float(df["pressure_hPa"].min()) if df["pressure_hPa"].notna().any() else np.nan  # pres mín ~ tope
        ztop = float(df["height_m"].max())     if df["height_m"].notna().any()     else np.nan  # altura máxima si existe
        meta.append({
            **hd, "catalog_key": key, "status": status, "n_levels": len(df),
            "n_valid_pressure": int(nP), "n_valid_temp": int(nT), "n_valid_wind": int(nW),
            "p_max_hPa_surface": pmax, "p_min_hPa_top": pmin, "z_top_m": ztop
        })                                                  # guarda una fila de metadatos por sondeo

    meta = pd.DataFrame(meta).sort_values("time").reset_index(drop=True)  # DataFrame ordenado temporalmente
    if drop_empty:
        # Si se pide, elimina sondeos "empty" tanto del catálogo como de meta (limpia el dataset)
        keep = set(meta[meta["status"]!="empty"]["catalog_key"])          # claves a conservar (no vacíos)
        cat  = {k:v for k,v in cat.items() if k in keep}                  # filtra el dict catálogo
        meta = meta[meta["status"]!="empty"].reset_index(drop=True)       # filtra el DataFrame meta
    return cat, meta                                                      # devuelve: (catálogo de perfiles, tabla de meta)


In [6]:
# ============================================================
# Celda 5: descarga de estación (ZIP en memoria) + versión con caché local
# - Cada estación IGRA tiene un archivo ZIP con TODO su "period of record" (POR).
# - Aquí definimos:
#     (1) Un descargador "rápido" que baja el ZIP a memoria y lo lee sin escribir a disco.
#     (2) Un descargador con CACHÉ local para no re-descargar el mismo ZIP si ya existe.
# - El resultado es un iterador de líneas: así los archivos de ~300 MB no se duplican en RAM.
# ============================================================

def download_igra_station_file(station_id, base="https://www.ncei.noaa.gov/pub/data/igra/data/data-por"):
    # Construye la URL completa al ZIP de la estación (POR = period-of-record).
    # Ejemplo: https://.../data-por/PSM00091408-data.txt.zip
    url = f"{base}/{station_id}-data.txt.zip"
    # Descarga el ZIP completo en memoria (objeto requests.Response) con un timeout de 120 s.
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    # Abre el contenido binario como un archivo ZIP directamente en memoria (sin escribir a disco).
    z = zipfile.ZipFile(io.BytesIO(r.content))
    # IGRA contiene un único TXT; se entrega línea por línea para limitar memoria.
    fname = z.namelist()[0]
    with z.open(fname, "r") as src:
        for raw in src:
            yield raw.decode("utf-8", errors="replace").rstrip("\r\n")

# Carpeta para CACHÉ local de ZIPs: ahorra ancho de banda y acelera iteraciones
CACHE_DIR = "./data/IGRA/cache_zip"
# Crea la carpeta si no existe (no falla si ya existe).
os.makedirs(CACHE_DIR, exist_ok=True)

def download_igra_station_file_cached(station_id, base="https://www.ncei.noaa.gov/pub/data/igra/data/data-por"):
    # Ruta en disco donde guardaremos (o ya tenemos) el ZIP de esta estación.
    zpath = os.path.join(CACHE_DIR, f"{station_id}-data.txt.zip")
    # Si NO existe en caché, lo descargamos ahora y lo guardamos a disco.
    if not os.path.exists(zpath):
        # Construye la URL igual que arriba (POR de la estación).
        url = f"{base}/{station_id}-data.txt.zip"
        # Descarga con timeout para robustez.
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        # Escribe el contenido binario (ZIP) al archivo local de caché.
        with open(zpath, "wb") as f:
            f.write(r.content)
    # Abre SIEMPRE desde caché (sea recién descargado o existente de antes).
    with zipfile.ZipFile(zpath, "r") as z:
        # Tomamos el único .txt dentro del ZIP.
        fname = z.namelist()[0]
        # Iteración streaming: el contexto permanece abierto hasta consumir el generador.
        with z.open(fname, "r") as src:
            for raw in src:
                yield raw.decode("utf-8", errors="replace").rstrip("\r\n")


In [7]:
# ============================================================ 
# Celda 6: selección reproducible de las 2 estaciones
# - Criterio útil para ESTE trabajo: T(p) debe cubrir 980–850 hPa, lo necesario para N².
# - La selección validada queda congelada en data/station_selection.csv para que una corrida
#   normal no vuelva a descomprimir 40 archivos ni cambie cuando NOAA actualice sus POR.
# - Para auditar/recalcular el ranking: CLIMA_RECOMPUTE_SELECTION=1.
# ============================================================

TOP_BY_NOBS = 40     # evalúo las 40 con más NOBS en la caja MJO
#   (NOBS = número total de lanzamientos) dentro de la caja MJO.

SELECTION_SNAPSHOT_PATH = "./data/station_selection.csv"
RECOMPUTE_SELECTION = os.environ.get("CLIMA_RECOMPUTE_SELECTION", "0") == "1"

candidates = mjo_stations.sort_values("nobs", ascending=False).head(TOP_BY_NOBS).copy()
#  De las estaciones dentro de la caja MJO (Celda 2), ordena por NOBS y toma las TOP_BY_NOBS.

def score_station_streaming(lines, p_bottom=980.0, p_top=850.0):
    """Cuenta sondeos cuya temperatura válida abarca ambos extremos de la capa N²."""
    total = useful = 0
    in_sounding = False
    pmin_t, pmax_t = np.inf, -np.inf

    def close():
        nonlocal total, useful, in_sounding, pmin_t, pmax_t
        if in_sounding:
            total += 1            # contamos el sondeo que se cierra
            useful += int(pmin_t <= p_top and pmax_t >= p_bottom)
        in_sounding = False
        pmin_t, pmax_t = np.inf, -np.inf

    for ln in lines:
        if not ln.strip():
            continue              # ignora líneas en blanco
        if ln.startswith("#"):
            close()               # cierra el sondeo anterior (si había) y abre uno nuevo
            in_sounding = True
            continue
        if not in_sounding:
            continue              # seguridad: si apareció línea suelta, se ignora

        p_pa = _to_int_field(ln[9:15])
        t_t  = _to_int_field(ln[22:27])
        if p_pa not in (None, *QC_REMOVE) and t_t not in (None, *QC_REMOVE):
            p_hPa = p_pa / 100.0
            pmin_t, pmax_t = min(pmin_t, p_hPa), max(pmax_t, p_hPa)

    close()  # cierra el último sondeo si el archivo no termina en '#'

    frac = useful/total if total else 0.0
    return {"total": total, "useful": useful, "frac_useful": frac}
    #  Devolvemos conteos y fracción; 'frac_useful' será la métrica para ordenar estaciones.

if RECOMPUTE_SELECTION:
    scores = []
    for _, row in candidates.iterrows():
        sid = row["id"]
        try:
            met = score_station_streaming(download_igra_station_file_cached(sid))
            scores.append({**row.to_dict(), **met})
        except Exception as e:
            print("[WARN]", sid, e)
    scores_df = pd.DataFrame(scores).sort_values(["frac_useful", "useful"], ascending=False).reset_index(drop=True)
    chosen = scores_df.head(2).copy()
    print("Top 10 recalculado por cobertura T(p) de 980–850 hPa:")
    display(scores_df.head(10)[["id","name","lat","lon","nobs","total","useful","frac_useful"]])
else:
    selection = pd.read_csv(SELECTION_SNAPSHOT_PATH)
    chosen = selection.merge(stations, on="id", how="left", validate="one_to_one")
    chosen["id"] = pd.Categorical(chosen["id"], categories=selection["id"], ordered=True)
    chosen = chosen.sort_values("id").reset_index(drop=True)
    chosen["id"] = chosen["id"].astype(str)
    scores_df = chosen.copy()
    print("Selección congelada y reproducible (para recalcular: CLIMA_RECOMPUTE_SELECTION=1):")
    display(chosen[["id","name","lat","lon","nobs","total","useful","frac_useful"]])

display(chosen)
# Estas 2 estaciones serán las "ganadoras" para el resto del análisis:
#   descarga completa, construcción de perfiles θ, cálculo de dθ/dz(0–3 km), N²(980–850 hPa),
#   construcción de anomalías diarias (climatología 1981–2010), y compositing por fase/grupo RMM.


Selección congelada y reproducible (para recalcular: CLIMA_RECOMPUTE_SELECTION=1):


,id,name,lat,lon,nobs,total,useful,frac_useful
0,PSM00091408,KOROR/CAROLINE IS.,7.3333,134.4833,49500,49512,44568,0.900145
1,FMM00091334,TRUK/CAROLINE IS.,7.4500,151.8333,48560,48573,43530,0.896177


,id,total,useful,frac_useful,criterion,input_snapshot,lat,lon,elev_m,state,name,fstyear,lstyear,nobs
0,PSM00091408,49512,44568,0.900145,T(p) covers 980-850 hPa,IGRA cache 2025-09-05,7.3333,134.4833,30.0,NaN,KOROR/CAROLINE IS.,1950,2025,49500
1,FMM00091334,48573,43530,0.896177,T(p) covers 980-850 hPa,IGRA cache 2025-09-05,7.4500,151.8333,3.0,NaN,TRUK/CAROLINE IS.,1951,2025,48560


In [8]:
# ============================================================
# Celda 8: termodinámica básica (θ) y utilidades
# - Definir constantes físicas y funciones auxiliares para:
#   (a) calcular temperatura potencial θ (Poisson),
#   (b) ordenar perfiles por presión (superficie→tope),
#   (c) interpolar temperatura a niveles isobáricos fijos (p. ej. 1000 y 850 hPa),
# lo cual es requisito para construir perfiles de θ, dθ/dz y N² en la capa baja.
# ============================================================

# Constantes físicas
G   = 9.80665        # m s^-2  | Gravedad estándar (se usa en N², hypsométrica; aquí queda por coherencia general)
RD  = 287.05         # J kg^-1 K^-1 | Constante de gas del aire seco (R_d)
CP  = 1004.0         # J kg^-1 K^-1 | Calor específico a presión constante (c_p) del aire seco
KAPPA = RD/CP        # adim.    | Exponente de Poisson (κ = R_d / c_p) para θ
P0  = 1000.0         # hPa      | Presión de referencia para θ (nivel estándar)

def theta_potential_temperature(T_K, p_hPa):
    """Poisson: θ = T * (p0/p)^kappa"""
    # T_K: temperatura absoluta (Kelvin) en el nivel
    # p_hPa: presión (hPa) del mismo nivel
    # Devuelve la temperatura potencial θ (K), que “traslada” el aire a 1000 hPa
    # sin intercambio de calor, útil para diagnosticar estabilidad y mezcla.
    return T_K * (P0 / p_hPa) ** KAPPA

def _safe_sort_p(df):
    """Ordena por presión descendente si existe."""
    d = df.copy()
    # Si hay al menos algún valor válido de presión, ordena de mayor a menor:
    # esto coloca la “superficie” (p más alta) arriba y el “tope” (p menor) abajo,
    if d["pressure_hPa"].notna().any():
        return d.sort_values("pressure_hPa", ascending=False).reset_index(drop=True)
    # Si no hay presión disponible, devuelve tal cual (sin ordenar).
    return d

def _interp_T_at_pressure(p_hPa, T_K, target_hPa):
    """Interpolación lineal T(p) en presión (hPa)."""
    # p_hPa: array/serie de presiones (hPa)
    # T_K:   array/serie de temperaturas (K) asociadas a esas presiones
    # target_hPa: presión objetivo (hPa) donde queremos estimar T por interpolación
    p = np.asarray(p_hPa, float)   # Convierte a ndarray float para robustez numérica
    T = np.asarray(T_K, float)     # Idem para T
    # Máscara de valores finitos (descarta NaN/inf); exige al menos 2 puntos válidos para interpolar.
    m = np.isfinite(p) & np.isfinite(T)
    if m.sum() < 2:
        return np.nan
    # Verifica que el nivel objetivo esté dentro del rango de presiones disponibles.
    # La interpolación lineal simple no extrapola; por eso retornamos NaN si está fuera.
    if not (np.nanmin(p[m]) <= target_hPa <= np.nanmax(p[m])):
        return np.nan
    # Interpola en presión:
    # np.interp espera x creciente; como los perfiles suelen venir en p descendente (superficie→tope),
    # invertimos el orden [::-1] para que el eje de presiones sea creciente y la interpolación sea correcta.
    # Devuelve temperatura interpolada (float) en K en el nivel isobárico target_hPa.
    return float(np.interp(target_hPa, p[m][::-1], T[m][::-1]))  # p desc → invierto para orden creciente


In [9]:
# ============================================================ 
# Celda 9: N² (bulk) 980–850 hPa por sondeo 
# - Calcular una aproximación "bulk" (de capa) de la estabilidad estática (N²) entre 980 y 850 hPa
#   para CADA sondeo. Esta métrica resume la estratificación de la capa baja y se usa luego para:
#   * construir series diarias, climatologías (1981–2010) y anomalías,
#   * hacer composites por fase/grupo RMM (preacondicionamiento del MJO).
# Fundamento:
#   N² ≈ (g/θ̄) * (Δθ/Δz), donde:
#     - θ_980 y θ_850 se obtienen de T interpolada a 980 y 850 hPa (Poisson),
#     - Δz se estima con la ecuación hipsométrica seca usando T̄ entre 980–850 hPa,
#     - Unidades finales: s^-2 (N²>0 = estable; N²<0 = inestable).
# ============================================================

def bulk_N2_between_pressures(df_profile, p_bottom, p_top):
    """
    Implementa:
      N² ≈ (g / θ̄) * (θ_top - θ_bottom) / Δz
    con Δz = (R_d * T̄ / g) * ln(p_bottom/p_top) (aproximación seca).
    y θ_p = T_p * (p0/p)^kappa.

    Devuelve N² en s^-2.
    """
    # 1) Asegura orden lógico del perfil (superficie→tope) y exige que existan p y T.
    #    _safe_sort_p ordena por presión descendente si hay presión disponible.
    d = _safe_sort_p(df_profile.dropna(subset=["pressure_hPa","temp_C"]))
    if d.empty:
        # Si no hay datos suficientes de p y T en el sondeo, no se puede calcular N² → NaN.
        return np.nan

    # 2) Prepara vectores de T (en Kelvin) y p (en hPa) para interpolar a 980 y 850 hPa.
    T = d["temp_C"].values + 273.15   # convierte °C → K (Poisson y hipsométrica requieren K)
    p = d["pressure_hPa"].values      # presión en hPa ya normalizada en el parser

    # 3) Interpolación lineal de T a niveles isobáricos estándar.
    #    _interp_T_at_pressure garantiza que el target esté dentro del rango de p válidas.
    T_bottom = _interp_T_at_pressure(p, T, float(p_bottom))
    T_top = _interp_T_at_pressure(p, T, float(p_top))
    if not np.isfinite(T_bottom) or not np.isfinite(T_top):
        # Si no se puede interpolar (por ejemplo, el perfil no cubre 980–850 hPa), devuelve NaN.
        return np.nan

    # 4) Calcula θ en los extremos (Poisson) y su media θ̄ para la fórmula de N².
    th_bottom = theta_potential_temperature(T_bottom, float(p_bottom))
    th_top = theta_potential_temperature(T_top, float(p_top))
    thbar = 0.5 * (th_bottom + th_top)

    # 5) Estima el espesor Δz con la ecuación hipsométrica seca usando T̄ (media entre 980 y 850 hPa).
    #    Δz = (R_d * T̄ / g) * ln(p980/p850) → unidades: metros.
    Tbar = 0.5 * (T_bottom + T_top)
    dz = (RD * Tbar / G) * np.log(float(p_bottom) / float(p_top))
    if dz <= 0:
        # Control de sanidad: Δz debe ser positivo (p980 > p850).
        return np.nan

    # 6) Aplica la fórmula bulk de N² con Δθ/Δz (θ en K, z en m). Resultado en s^-2.
    N2 = (G / thbar) * ((th_top - th_bottom) / dz)

    # 7) Devuelve como float estándar (no numpy scalar) para facilitar persistencia/plots.
    return float(N2)

def bulk_N2_980_850(df_profile):
    return bulk_N2_between_pressures(df_profile, 980.0, 850.0)

def bulk_N2_1000_850(df_profile):
    return bulk_N2_between_pressures(df_profile, 1000.0, 850.0)


In [10]:
# ============================================================ 
# Celda 9-bis: serie diaria de N²_bulk para una estación y merge con RMM
# - Calcula, para UNA estación IGRA, la métrica de estabilidad estática "bulk"
#   N² (980–850 hPa) por CADA sondeo (Celda 9) y luego la agrega a resolución DIARIA.
# - Finalmente, une esa serie diaria con el índice RMM (fase, amplitud) por fecha
#   para poder hacer histogramas/boxplots y composites condicionados a la fase MJO.
# ============================================================

def station_daily_N2_bulk(station_id, start="1980-01-01", end=None):
    """Calcula N²_bulk(980–850) por sondeo y promedia a diario. Devuelve DF unido con RMM por fecha."""
    #  Firma de la función:
    #   - station_id: código IGRA de la estación (p.ej., 'PSM00091408').
    #   - start, end: límites temporales opcionales (string ISO) para filtrar el periodo.
    #   - Return: DataFrame con columnas de N2_s2 diario y columnas RMM (rmm1, rmm2, phase, amplitude, group).
    
    lines = download_igra_station_file_cached(station_id)
    #  El ZIP se recorre en streaming: cada perfil se descarta después de calcular su N².

    start_ts = pd.Timestamp(start) if start else None
    end_ts   = pd.Timestamp(end)   if end   else None
    #  Convierte los límites temporales (si se proporcionan) a Timestamp para comparaciones eficientes.

    rows = []
    #  Acumulará una lista de dicts con N² por sondeo y su fecha (luego promediaremos por día).

    for header, block in iter_soundings(lines):
        hd = parse_header_line(header)
        ts = pd.to_datetime(hd["time"])
        
        if start_ts and ts < start_ts: 
            continue
        if end_ts and ts >= end_ts:
            continue
        #  Filtrado temporal: si 'start'/'end' están definidos, ignora sondeos fuera del rango.

        level_rows = [parse_level_line(ln) for ln in block]
        if not level_rows:
            continue
        df = pd.DataFrame(level_rows)
        n2 = bulk_N2_980_850(df)
        #  Calcula N² "bulk" (s^-2) entre 980 y 850 hPa para ESTE sondeo (ver Celda 9).
        #   Si el perfil no cubre 980–850 hPa o faltan T/p, devolverá NaN.

        rows.append({"station": station_id, "time": ts, "date": ts.floor("D"), "N2_s2": n2})
        # ^ Guarda un registro por sondeo con:
        #   - 'station': id de la estación (para agrupar después),
        #   - 'time'   : fecha-hora exacta del sondeo,
        #   - 'date'   : fecha diaria (sin hora) para poder promediar en el día,
        #   - 'N2_s2'  : valor de N² (s^-2) del sondeo.

    if not rows:
        #  Si no hubo sondeos (o todos dieron NaN/filtrados), devuelve un DataFrame vacío
        #   con el esquema esperado (incluye las columnas que vendrán del merge con RMM).
        return pd.DataFrame(columns=["station","time","date","N2_s2","rmm1","rmm2","phase","amplitude","group"])

    daily = (pd.DataFrame(rows)
             .dropna(subset=["N2_s2"])
             .groupby(["station","date"], as_index=False)["N2_s2"].mean())
    #  Convierte la lista 'rows' en DataFrame, descarta sondeos con N2_s2=NaN,
    #   agrupa por estación y fecha diaria, y calcula la MEDIA diaria de N².
    #   Resultado: una fila por (station, date) con 'N2_s2' diario.

    # Une con RMM por fecha (se define en Celdas 12)
    merged = daily.merge(rmm[["date","rmm1","rmm2","phase","amplitude","group"]],
                         on="date", how="left")
    #  Join por la columna 'date' con el DataFrame global 'rmm' (índice diario RMM).
    #   Añade:
    #   - rmm1, rmm2: componentes del índice de Wheeler–Hendon,
    #   - phase     : fase 1–8 del MJO (entera),
    #   - amplitude : módulo (actividad MJO: >1 activo),
    #   - group     : agrupación 1-8, 2-3, 4-5, 6-7 (útil para tus boxplots y composites).
    #   how="left" conserva todas tus fechas con N², aunque falte RMM en alguna (quedará NaN).

    return merged.sort_values(["station","date"]).reset_index(drop=True)
    #  Ordena por estación y fecha para series limpias y resetea el índice.


In [11]:
# ============================================================
# Celda 10–12: descarga y carga del RMM (BoM/NOAA) + grupos 1-8/2-3/4-5/6-7
# - RMM (Wheeler–Hendon) es el índice diario del MJO: entrega rmm1, rmm2, fase (1–8) y amplitud.
# - Aquí:
#   1) Garantizamos que exista un archivo local "rmm.74toRealtime.txt" (lo bajamos de BoM o NOAA-CPC).
#   2) Lo parseamos a DataFrame (saltando encabezados/valores missing 999 o ~1e36).
#   3) Recortamos desde 1980 para empatar con tus series de radiosondeos.
#   4) Mapemos fase→grupo (1-8 / 2-3 / 4-5 / 6-7) para tus boxplots/composites.
# ============================================================

RMM_LOCAL_PATH = "data/rmm.74toRealtime.txt"              # ruta donde esperamos tener el archivo RMM (texto plano)
os.makedirs(os.path.dirname(RMM_LOCAL_PATH), exist_ok=True)  # crea carpeta 'data/' si no existe (evita errores de I/O)

def fetch_rmm_any(save_to=RMM_LOCAL_PATH) -> str:
    #  Intenta descargar el archivo RMM desde 2 fuentes oficiales:
    #   - BoM (Australia)
    #   - NOAA-CPC (EE.UU., espejo)
    # Si la primera falla, prueba la segunda. Devuelve la ruta local guardada.
    sources = [
        ("BoM", "https://www.bom.gov.au/climate/mjo/graphics/rmm.74toRealtime.txt"),
        ("NOAA-CPC", "https://www.cpc.ncep.noaa.gov/products/precip/CWlink/daily_mjo_index/proj.RMM.74toRealtime.txt"),
    ]
    headers = {"User-Agent": "Mozilla/5.0"}  # user-agent simple para evitar rechazos (algunos servidores bloquean clientes vacíos)
    last_err = None
    for name, url in sources:
        try:
            r = requests.get(url, timeout=120, headers=headers)  # descarga con timeout para robustez
            r.raise_for_status()                                 # si HTTP != 200, lanza excepción
            with open(save_to, "wb") as f:
                f.write(r.content)                                # guarda el archivo en disco (binario)
            print(f"RMM descargado desde {name}.")
            return save_to                                        # retorno la ruta del archivo listo para leer
        except Exception as e:
            last_err = f"{name}: {e}"                             # guarda último error para informar si ambas fallan
    raise IOError(f"No se pudo descargar RMM. Último error: {last_err}")  # error claro si no hubo éxito

def load_rmm_from_file(path: str) -> pd.DataFrame:
    #  Lee el archivo RMM local (texto) y lo convierte en DataFrame ordenado por fecha.
    rows = []
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for raw in f:
            line = raw.strip()
            if not line: 
                continue                                          # salta líneas vacías
            c0 = line[0]
            # El archivo trae encabezados/comentarios; conservamos solo líneas que empiezan con año
            if not (c0.isdigit() or (c0=="-" and len(line)>1 and line[1].isdigit())):
                continue
            parts = line.split()
            if len(parts) < 7:
                continue                                          # líneas válidas deben tener al menos 7 columnas
            try:
                y,m,d = int(parts[0]), int(parts[1]), int(parts[2])          # fecha (YYYY MM DD)
                r1,r2,ph,amp = float(parts[3]), float(parts[4]), int(parts[5]), float(parts[6])  # rmm1, rmm2, fase, amplitud
            except:
                continue                                          # si algún parse falla, ignora la línea
            # Missing flags documentados en el header del archivo: 999 o ~1e36 (1.E36)
            if any([(v==999) for v in (r1,r2,amp)]) or any([abs(v)>=1e35 for v in (r1,r2,amp)]):
                continue
            rows.append({"date": datetime(y,m,d), "rmm1": r1, "rmm2": r2, "phase": ph, "amplitude": amp})
    if not rows:
        raise ValueError("RMM vacío/ilegible.")                    # sanity check si no se pudo leer nada útil
    df = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)  # ordena por fecha ascendente
    return df

def load_rmm(local_path=RMM_LOCAL_PATH) -> pd.DataFrame:
    #  Cargador “alto nivel”: si no existe el archivo local, lo descarga; luego lo parsea.
    if not os.path.exists(local_path):
        fetch_rmm_any(local_path)
    return load_rmm_from_file(local_path)

# Carga y recorte desde 1980 (para empatar con series)
rmm = load_rmm(RMM_LOCAL_PATH)                                        # DataFrame con columnas: date, rmm1, rmm2, phase, amplitude
rmm = rmm[rmm["date"] >= pd.Timestamp("1980-01-01")].reset_index(drop=True)  # recorte temporal (consistencia con radiosondeos)

# Mapeo de fase a grupos
def rmm_group_from_phase(phase: int) -> str:
    # ^ Agrupa las 8 fases en 4 grupos regionales (como pidió el profe):
    #   1-8 = Western Hemisphere & Africa
    #   2-3 = Indian Ocean
    #   4-5 = Maritime Continent
    #   6-7 = Western Pacific
    if phase in (1,8): return "1-8"
    if phase in (2,3): return "2-3"
    if phase in (4,5): return "4-5"
    if phase in (6,7): return "6-7"
    return "NA"  # por seguridad, etiqueta 'NA' si no es 1–8

rmm["group"] = rmm["phase"].apply(lambda p: rmm_group_from_phase(int(p)) if pd.notna(p) else "NA")
# Crea la columna 'group' con el grupo regional según la fase (útil para boxplots y composites).
rmm.head(10)

,date,rmm1,rmm2,phase,amplitude,group
0,1980-01-01,0.811586,-0.066468,4,0.814303,4-5
1,1980-01-02,0.705088,-0.029684,4,0.705712,4-5
2,1980-01-03,0.662733,0.037199,5,0.663776,4-5
3,1980-01-04,0.614840,0.213630,5,0.650896,4-5
4,1980-01-05,0.702877,0.490769,5,0.857257,4-5
5,1980-01-06,0.758704,0.604175,5,0.969876,4-5
6,1980-01-07,0.674902,0.379001,5,0.774037,4-5
7,1980-01-08,0.764337,0.197338,5,0.789400,4-5
8,1980-01-09,0.795880,0.209051,5,0.822878,4-5
9,1980-01-10,0.937358,0.256808,5,0.971901,4-5


In [12]:
# ============================================================
# Celda 13: construir DF maestro (dos estaciones) y guardar (Parquet si hay motor; si no CSV)
# ============================================================

# Toma los IDs de las dos estaciones seleccionadas en la Celda 6 
sid1, sid2 = chosen["id"].iloc[0], chosen["id"].iloc[1]
print("Estaciones elegidas:", sid1, "y", sid2)

# Calcula la serie diaria para cada estación:
#   - `station_daily_N2_bulk` lee todos los sondeos IGRA de la estación,
#     calcula N²_bulk (980–850 hPa) por sondeo, promedia a diario,
#     y UNE por fecha con el índice RMM (fase, amplitud, grupo).
# Luego concatena las dos estaciones en un único DataFrame.
daily_2st = pd.concat(
    [station_daily_N2_bulk(sid1, start="1980-01-01"),
     station_daily_N2_bulk(sid2, start="1980-01-01")],
    ignore_index=True
)

# Muestra tamaño (filas, columnas) y las primeras filas para verificar que
# están las columnas esperadas: ['station','date','N2_s2','rmm1','rmm2','phase','amplitude','group'].
print("Shape diario (con RMM):", daily_2st.shape)
display(daily_2st.head())

# Crea carpeta de salida donde se guardará el archivo con los resultados diarios.
os.makedirs("./outputs", exist_ok=True)

# Escribe SIEMPRE el CSV y, si hay motor disponible, también Parquet. Así nunca quedan
# dos archivos con resultados de corridas distintas (problema detectado en la auditoría).
daily_2st.to_csv("./outputs/daily_2stations.csv", index=False)
print("Guardado ./outputs/daily_2stations.csv")
try:
    daily_2st.to_parquet("./outputs/daily_2stations.parquet", index=False)
    print("Guardado ./outputs/daily_2stations.parquet")
except Exception as e:
    print("Parquet no disponible; el CSV ya quedó guardado:", e)


Estaciones elegidas: PSM00091408 y FMM00091334


Shape diario (con RMM): (33096, 8)


,station,date,N2_s2,rmm1,rmm2,phase,amplitude,group
0,PSM00091408,1980-01-01,0.000097,0.811586,-0.066468,4.0,0.814303,4-5
1,PSM00091408,1980-01-02,0.000133,0.705088,-0.029684,4.0,0.705712,4-5
2,PSM00091408,1980-01-03,0.000112,0.662733,0.037199,5.0,0.663776,4-5
3,PSM00091408,1980-01-04,0.000121,0.614840,0.213630,5.0,0.650896,4-5
4,PSM00091408,1980-01-05,0.000168,0.702877,0.490769,5.0,0.857257,4-5


Guardado ./outputs/daily_2stations.csv
Guardado ./outputs/daily_2stations.parquet


In [13]:
# ============================================================ 
# Celda 14: HISTOGRAMAS de N² por fase (1–8) — una figura por estación
#  * Fix: tema oscuro local + texto blanco y líneas claras
# ============================================================
def plot_histograms_N2_by_phase(master_df, station_id, bins=30):
    # Filtra el DataFrame maestro para una estación específica y descarta N² faltantes.
    df = master_df[(master_df["station"] == station_id) & (master_df["N2_s2"].notna())].copy()
    if df.empty:
        print("Sin datos para", station_id)
        return

    # Rango común (1–99 percentil) para todas las subtramas
    xmin, xmax = np.nanpercentile(df["N2_s2"], [1, 99])

    phases = list(range(1, 9))

    # --- Estilo oscuro LOCAL para que no herede rcParams globales ---
    dark_rc = {
        "figure.facecolor": "#000000",
        "axes.facecolor":   "#000000",
        "savefig.facecolor":"#000000",
        "axes.edgecolor":   "#B0B0B0",
        "axes.labelcolor":  "white",
        "axes.titlecolor":  "white",
        "xtick.color":      "white",
        "ytick.color":      "white",
        "grid.color":       "#666666",
        "grid.alpha":       0.35,
    }

    with plt.rc_context(dark_rc):
        fig, axes = plt.subplots(2, 4, figsize=(15, 5), sharex=True, sharey=True)
        axes = axes.ravel()

        for i, ph in enumerate(phases):
            ax = axes[i]
            vals = df.loc[df["phase"] == ph, "N2_s2"].dropna().values

            # Histograma con bordes claros (visibles en negro)
            ax.hist(
                vals,
                bins=bins,
                range=(xmin, xmax),
                alpha=0.85,
                color="#4DA3FF",                  # azul brillante (visible)
                edgecolor="#E6E6E6",              # borde claro
                linewidth=0.6
            )

            # Línea de referencia en 0 (gris claro sobre negro)
            ax.axvline(0, color="#BBBBBB", lw=1.0)

            # Títulos y etiquetas visibles
            ax.set_title(f"Fase {ph}  (n={len(vals)})", fontsize=14, color="white")
            if i % 4 == 0:
                ax.set_ylabel("Frecuencia", fontsize=14, color="white")
            if i >= 4:
                ax.set_xlabel("N² (s$^{-2}$)", fontsize=14, color="white")

            ax.tick_params(colors="white", labelsize=12)
            for sp in ax.spines.values():
                sp.set_color("#B0B0B0")
            ax.grid(True)

        # Título general
        fig.suptitle(f"{station_id} · Histograma N² por fase RMM (980–850 hPa)",
                     color="white", fontsize=16)
        fig.tight_layout(rect=[0, 0, 1, 0.95])
        plt.show()

# Ejecutar para las estaciones del DataFrame diario
for sid in daily_2st["station"].unique():
    plot_histograms_N2_by_phase(daily_2st, sid, bins=30)


C:\Users\Camilo\AppData\Local\Temp\ipykernel_4784\2758719074.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\Camilo\AppData\Local\Temp\ipykernel_4784\2758719074.py:69: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
# ============================================================
# Celda 14-c: Histograma unificado INTERACTIVO (8 fases RMM)
# ============================================================
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from ipywidgets import interact, widgets

# Render en notebooks (ajusta si tu entorno requiere otro)
pio.renderers.default = "plotly_mimetype"  # o "notebook_connected"

def unified_histogram_plotly(df, station, bins=40, density=True, amp_min=1.0):
    """
    Histograma unificado de N² (980–850 hPa) por fases RMM (1..8),
    superpuestos con misma escala de bins.
    """
    d = df[(df["station"] == station) & df["N2_s2"].notna()].copy()
    if amp_min is not None:
        d = d[(d["amplitude"].notna()) & (d["amplitude"] >= amp_min)]
    if d.empty:
        return px.histogram(title=f"{station} · N² (980–850 hPa) — sin datos con el filtro actual")

    # Rango común (percentiles 1–99) para que no lo distorsionen outliers
    xmin, xmax = np.nanpercentile(d["N2_s2"], [1, 99])
    histnorm = "probability density" if density else None  # densidad vs conteo

    # Fase como categoría ordenada (1..8) y color
    d["phase_str"] = d["phase"].astype(int).astype(str)
    phase_order = [str(i) for i in range(1, 9)]

    fig = px.histogram(
        d, x="N2_s2", color="phase_str",
        nbins=bins, histnorm=histnorm,
        opacity=0.35, barmode="overlay",
        category_orders={"phase_str": phase_order},
        range_x=[xmin, xmax],
        labels={"N2_s2": "N² (×10⁻⁶ s⁻²)", "phase_str": "Fase RMM"},
        title=f"{station} · N² (980–850 hPa) (amp ≥ {amp_min:.1f})"
    )

    fig.update_traces(marker_line_width=1.2)  # bordes para que se distingan
    fig.update_layout(
        legend_title_text="Fase RMM",
        legend=dict(orientation="h", y=1.12, x=0.0),
        margin=dict(l=60, r=10, t=70, b=50),
        bargap=0.0
    )
    fig.update_yaxes(title_text="Densidad" if density else "Frecuencia")
    fig.update_xaxes(title_text="N² (s⁻²)")
    return fig

# ---------- UI interactiva ----------
_stations = sorted(daily_2st["station"].unique())

# En ejecución headless se define la función, pero no se intenta serializar el widget.
if os.environ.get("CLIMA_HEADLESS", "0") == "1":
    def _interactive_only(**_widget_args):
        return lambda func: func
else:
    _interactive_only = interact

@_interactive_only(
    station=widgets.Dropdown(options=_stations, description="Estación:", value=_stations[0]),
    amp_min=widgets.FloatSlider(value=1.0, min=0.0, max=2.5, step=0.1, description="amp ≥"),
    bins=widgets.IntSlider(value=40, min=10, max=80, step=2, description="bins"),
    density=widgets.ToggleButtons(options=[("Densidad", True), ("Conteo", False)], description="Eje Y:")
)
def _show(station, amp_min, bins, density):
    fig = unified_histogram_plotly(daily_2st, station, bins=bins, density=density, amp_min=amp_min)
    fig.show()


In [15]:
# ============================================================
# Celda 14-c (DARK+BIG FONTS): Histograma unificado interactivo
#  - Tema negro para figura y controles
#  - Tipografía GRANDE y legible en widgets
# ============================================================
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import ipywidgets as widgets
from IPython.display import display, HTML

pio.renderers.default = "plotly_mimetype"  # o "notebook_connected"

# ---- CSS: tema oscuro + FUENTE GRANDE en widgets ----
display(HTML("""
<style>
/* Contenedor oscuro de los controles */
.dark-panel {
  background:#000; color:#fff;
  padding:14px; border:1px solid #444; border-radius:12px;
  --font-lg: 18px;           /* tamaño base */
  --font-xl: 20px;           /* títulos/labels */
}
/* Labels (descriptions), textos y lecturas */
.dark-panel label,
.dark-panel .widget-label,
.dark-panel .widget-readout,
.dark-panel .widget-readout input {
  color:#fff !important;
  font-size: var(--font-xl) !important;
}
/* Inputs comunes (dropdown, cajas de texto, etc.) */
.dark-panel select,
.dark-panel input,
.dark-panel textarea {
  color:#fff !important; background:#111 !important;
  border:1px solid #555 !important;
  font-size: var(--font-lg) !important;
}
/* --- FIX: ToggleButtons con fuente grande, sin recorte --- */
.dark-panel .widget-toggle-buttons>div>button {
  background:#111 !important; color:#fff !important;
  border:1px solid #555 !important;
  font-size: 18px !important;        /* grande y legible */
  padding:12px 18px !important;       /* más espacio interno */
  min-height:46px !important;         /* asegura altura suficiente */
  line-height:1.2 !important;         /* evita que toque los bordes */
  box-sizing:border-box !important;
  overflow:visible !important;        /* por si el tema aplica recortes */
  border-radius:10px !important;
  margin-right:8px !important;
}
.dark-panel .widget-toggle-buttons>div>button[aria-pressed="true"] {
  background:#333 !important; border-color:#888 !important;
}
/* Sliders */
.dark-panel .widget-slider .noUi-target { background:#222 !important; border:1px solid #555 !important; }
.dark-panel .widget-slider .noUi-connect { background:#444 !important; }
.dark-panel .widget-slider .noUi-handle {
  background:#777 !important; border:1px solid #aaa !important;
  width:18px !important; height:18px !important;
}
</style>
"""))

# Paleta brillante por fase (1..8)
PHASE_COLORS = {
    "1": "#1f77b4", "2": "#ff7f0e", "3": "#2ca02c", "4": "#d62728",
    "5": "#9467bd", "6": "#17becf", "7": "#e377c2", "8": "#bcbd22",
}

def unified_histogram_plotly(df, station, bins=40, density=True, amp_min=1.0):
    d = df[(df["station"] == station) & df["N2_s2"].notna()].copy()
    if amp_min is not None:
        d = d[(d["amplitude"].notna()) & (d["amplitude"] >= amp_min)]
    if d.empty:
        return px.histogram(title=f"{station} · N² (980–850 hPa) — sin datos con el filtro actual")

    xmin, xmax = np.nanpercentile(d["N2_s2"], [1, 99])
    histnorm = "probability density" if density else None

    d["phase_str"] = d["phase"].astype(int).astype(str)
    order = [str(i) for i in range(1, 9)]

    fig = px.histogram(
        d, x="N2_s2", color="phase_str",
        nbins=bins, histnorm=histnorm,
        opacity=0.45, barmode="overlay",
        category_orders={"phase_str": order},
        color_discrete_map=PHASE_COLORS,
        range_x=[xmin, xmax],
        labels={"N2_s2":"N² (s⁻²)", "phase_str":"Fase RMM"},
        title=f"{station} · N² (980–850 hPa)" # "(amp ≥ {amp_min:.1f}"
    )

    # Tema oscuro del gráfico (sin tocar datos)
    grid = "rgba(255,255,255,0.15)"
    fig.update_traces(marker_line_width=0.8,
                      marker_line_color="rgba(255,255,255,0.35)")
    fig.update_layout(
        paper_bgcolor="black", plot_bgcolor="black",
        font=dict(family="Arial", size=18, color="white"),
        title_font=dict(size=26, color="white"),
        legend_title_text="Fase RMM",
        legend=dict(orientation="h", y=1.16, x=0.0, bgcolor="rgba(0,0,0,0)",
                    font=dict(size=16, color="white")),
        margin=dict(l=80, r=20, t=86, b=68),
        bargap=0.0,
    )
    fig.update_xaxes(title_text="N² (s⁻²)", showline=True, linecolor="white",
                     ticks="outside", tickcolor="white", tickwidth=1.2,
                     gridcolor=grid, zerolinecolor=grid,
                     title_font=dict(size=20, color="white"),
                     tickfont=dict(size=16, color="white"))
    fig.update_yaxes(title_text=("Densidad" if density else "Frecuencia"),
                     showline=True, linecolor="white",
                     ticks="outside", tickcolor="white", tickwidth=1.2,
                     gridcolor=grid, zerolinecolor=grid,
                     title_font=dict(size=20, color="white"),
                     tickfont=dict(size=16, color="white"))
    return fig

# --------- Controles (con fuente grande) ---------
_stations = sorted(daily_2st["station"].unique())

station_dd = widgets.Dropdown(options=_stations, value=_stations[0], description="Estación:")
station_dd.style = {'description_width': 'auto'}
station_dd.layout = widgets.Layout(width='320px')

amp_sl = widgets.FloatSlider(value=1.0, min=0.0, max=2.5, step=0.1,
                             description="amp ≥", readout_format=".2f")
amp_sl.style = {'description_width': '80px', 'handle_color': '#888'}
amp_sl.layout = widgets.Layout(width='320px')

bins_sl = widgets.IntSlider(value=40, min=10, max=80, step=2, description="bins")
bins_sl.style = {'description_width': '60px', 'handle_color': '#888'}
bins_sl.layout = widgets.Layout(width='280px')

y_toggle = widgets.ToggleButtons(
    options=[("Densidad", True), ("Conteo", False)],
    description="Eje Y:"
)
y_toggle.style = {'button_width': '160px'}
y_toggle.layout = widgets.Layout(width='300px', height='auto')  # da aire vertical

panel = widgets.VBox(
    [widgets.HBox([station_dd, amp_sl, bins_sl]), y_toggle],
    layout=widgets.Layout(width="100%")
)
panel.add_class("dark-panel")  # aplica CSS oscuro y grande

out = widgets.Output()

def _render(*_):
    with out:
        out.clear_output(wait=True)
        fig = unified_histogram_plotly(
            daily_2st,
            station=station_dd.value,
            bins=bins_sl.value,
            density=y_toggle.value,
            amp_min=amp_sl.value
        )
        fig.show()

for w in (station_dd, amp_sl, bins_sl, y_toggle):
    w.observe(_render, names='value')

if os.environ.get("CLIMA_HEADLESS", "0") == "1":
    print("Widgets interactivos omitidos en ejecución headless; los HTML se generan en la celda siguiente.")
else:
    display(panel, out)
    _render()


Widgets interactivos omitidos en ejecución headless; los HTML se generan en la celda siguiente.


In [16]:
# === HTML con FRAMES (ligero) para CodePen/Canva — 1 estación por archivo ===
# - Mantiene slider "amp ≥" animado con pocos valores (reduce tamaño)
# - Botones Densidad/Conteo corregidos (method="update" + histnorm "")
# - Redondea datos para bajar bytes sin afectar resultados

import numpy as np
import pandas as pd
import plotly.graph_objects as go

# Paleta por fase (1..8)
PHASE_COLORS = {
    "1": "#1f77b4", "2": "#ff7f0e", "3": "#2ca02c", "4": "#d62728",
    "5": "#9467bd", "6": "#17becf", "7": "#e377c2", "8": "#bcbd22",
}

def build_hist_one_station_frames(
    df,
    station_id,
    filename="hist_station.html",
    # pocos valores para el slider de amplitud → menos frames
    amp_values=(0.0, 1.0, 1.5, 2.0),
    amp_default=1.0,
    bins_default=40,
    bins_min=10, bins_max=80, bins_step=10
):
    # ---------- Selección y limpieza ----------
    d = df[(df["station"]==station_id) & df["N2_s2"].notna()
           & df["phase"].notna() & df["amplitude"].notna()].copy()
    d["phase"] = d["phase"].astype(int)

    if d.empty:
        raise ValueError(f"No hay datos para la estación {station_id}")

    # Rango común (1–99 percentil) para el eje X
    xmin, xmax = np.nanpercentile(d["N2_s2"], [1, 99])
    bin_size_default = float((xmax - xmin) / bins_default)

    # ---------- Trazas iniciales (amp_default) ----------
    fig = go.Figure()
    traces_per_station = 8  # 8 fases
    # Redondeo para reducir peso sin afectar resultados
    # (N² ~1e-4 → 7 decimales sobran; amplitud con 2 decimales)
    trace_info = []  # guardamos x_all y a_all por fase

    for phase in range(1, 9):
        m = (d["phase"] == phase)
        x_all = np.round(d.loc[m, "N2_s2"].to_numpy(), 7)
        a_all = np.round(d.loc[m, "amplitude"].to_numpy(), 2)

        # filtro inicial por amp_default
        x0 = x_all[a_all >= amp_default]
        trace_info.append({"phase": phase, "x_all": x_all, "a_all": a_all})

        fig.add_trace(go.Histogram(
            x=x0,
            name=str(phase),
            opacity=0.45,
            marker=dict(color=PHASE_COLORS[str(phase)],
                        line=dict(width=0.6, color="rgba(255,255,255,0.35)")),
            xbins=dict(start=float(xmin), end=float(xmax), size=bin_size_default),
            histnorm="probability density",   # arranca en Densidad
            visible=True
        ))

    # ---------- Frames (uno por valor de amp ≥) ----------
    frames = []
    for val in amp_values:
        data_for_frame = []
        for ti in trace_info:
            x_new = ti["x_all"][ti["a_all"] >= val]
            # Solo actualizamos 'x' (lo demás se hereda del trace)
            data_for_frame.append(go.Histogram(x=x_new))
        frames.append(go.Frame(name=f"amp{val:.1f}", data=data_for_frame))
    fig.frames = frames

    # ---------- Botones Densidad / Conteo (fix) ----------
    updatemenus = []
    buttons_y = [
        dict(
            label="Densidad",
            method="update",
            args=[
                {"histnorm": "probability density"},     # a todas las trazas
                {"yaxis.title.text": "Densidad"}          # cambia eje Y
            ],
        ),
        dict(
            label="Conteo",
            method="update",
            args=[
                {"histnorm": ""},                         # "" = conteo
                {"yaxis.title.text": "Conteo"}
            ],
        ),
    ]
    updatemenus.append(dict(
        type="buttons",
        buttons=buttons_y,
        direction="right",
        x=0.02, y=1.12,
        bgcolor="black", bordercolor="#555",
        font=dict(color="white", size=16),
        pad=dict(r=10, t=0, l=10, b=0)
    ))

    # ---------- Sliders ----------
    # Slider de amplitud → activa frames
    amp_steps = []
    for val in amp_values:
        amp_steps.append(dict(
            label=f"{val:.1f}",
            method="animate",
            args=[[f"amp{val:.1f}"],
                  {"mode": "immediate",
                   "frame": {"duration": 0, "redraw": True},
                   "transition": {"duration": 0}}]
        ))

    # Slider de bins → cambia tamaño del bin (restyle)
    bin_steps = []
    for b in range(bins_min, bins_max + 1, bins_step):
        size = float((xmax - xmin) / b)
        bin_steps.append(dict(
            label=str(b),
            method="restyle",
            args=[{"xbins.size": size}]
        ))

    # ---------- Layout oscuro + ejes ----------
    fig.update_layout(
        updatemenus=updatemenus,
        sliders=[
            dict(active=max(0, list(amp_values).index(amp_default) if amp_default in amp_values else 0),
                 steps=amp_steps,
                 x=0.02, xanchor="left", y=1.04, len=0.46,
                 currentvalue=dict(prefix="amp ≥ ", font=dict(color="white", size=16)),
                 bgcolor="black", tickcolor="white"),
            dict(active=int((bins_default - bins_min)/bins_step),
                 steps=bin_steps,
                 x=0.52, xanchor="left", y=1.04, len=0.46,
                 currentvalue=dict(prefix="bins = ", font=dict(color="white", size=16)),
                 bgcolor="black", tickcolor="white"),
        ],
        barmode="overlay",
        paper_bgcolor="black", plot_bgcolor="black",
        font=dict(family="Arial", size=18, color="white"),
        title=f"{station_id} · N² (980–850 hPa)",
        title_font=dict(size=26, color="white"),
        legend_title_text="Fase RMM",
        legend=dict(orientation="h", y=1.0, x=0.0, bgcolor="rgba(0,0,0,0)",
                    font=dict(size=15, color="white")),
        margin=dict(l=80, r=20, t=110, b=70),
    )
    grid = "rgba(255,255,255,0.15)"
    fig.update_xaxes(
        title_text="N² (s⁻²)", showline=True, linecolor="white",
        ticks="outside", tickcolor="white", tickwidth=1.2,
        gridcolor=grid, zerolinecolor=grid
    )
    fig.update_yaxes(
        title_text="Densidad", showline=True, linecolor="white",
        ticks="outside", tickcolor="white", tickwidth=1.2,
        gridcolor=grid, zerolinecolor=grid
    )

    # Escribe HTML (Plotly desde CDN → archivo liviano)
    fig.write_html(filename, include_plotlyjs="cdn", full_html=True)
    print(f"✅ HTML escrito: {filename}  |  Estación: {station_id}")

# ======================
# Genera los dos archivos
# ======================

# 1) PSM (cambia el ID si el DF usa otro)
build_hist_one_station_frames(
    daily_2st,
    station_id="PSM00091408",
    filename="Histogra_PSM.html",
    amp_values=(0.0, 1.0, 1.5, 2.0),  # pocos frames → archivo chico
    amp_default=1.0,
    bins_default=40
)

# 2) FMM
build_hist_one_station_frames(
    daily_2st,
    station_id="FMM00091334",
    filename="Histogra_FMM.html",
    amp_values=(0.0, 1.0, 1.5, 2.0),
    amp_default=1.0,
    bins_default=40
)

# 3) HTML unificado que corresponde a los archivos publicados/abiertos en el IDE.
from clima_outputs import build_unified_histogram_html
_unified_histogram = build_unified_histogram_html(
    daily_2st,
    site_filename="hist-unificado-site/hist_unificado.html",
    codepen_filename="index_codepen.html",
    amp_default=1.0,
    bins_default=40,
)


✅ HTML escrito: Histogra_PSM.html  |  Estación: PSM00091408
✅ HTML escrito: Histogra_FMM.html  |  Estación: FMM00091334


HTML autocontenido: hist-unificado-site\hist_unificado.html
HTML para CodePen/CDN: index_codepen.html


In [17]:
# ============================================================
# Celda 15 (dark, sin relleno): BOXplots horizontales (1-8, 2-3, 4-5, 6-7) de N²
#   - Filtro: amplitud del MJO ≥ amp_min (por defecto 1.0)
#   - Eje X mostrado en micro (µ s^-2) sin alterar los datos
# ============================================================
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patheffects as pe
from matplotlib.ticker import FuncFormatter, MaxNLocator

# --- alta resolución para pantalla/guardado ---
plt.rcParams["figure.dpi"] = 180       # HD en notebook/preview
plt.rcParams["savefig.dpi"] = 450      # HD al exportar

def _micro_formatter(x, pos):
    """Convierte s^-2 a micro s^-2 en la etiqueta del eje (sin cambiar los datos)."""
    return f"{x*1e6:.0f}"  # 0.00010 -> '100'

def horizontal_boxplots_N2(master_df, station_id, amp_min=1.0, savepath=None):
    """
    Boxplots de N² (s^-2) por grupos de fase RMM (1-8, 2-3, 4-5, 6-7),
    filtrando por amplitud del MJO >= amp_min (por defecto 1.0).
    """
    # ---- 1) Selección y filtro ----
    df = master_df[(master_df["station"] == station_id) & (master_df["N2_s2"].notna())].copy()

    # Filtro de amplitud (MJO activo). Si no quieres filtrar, pasa amp_min=None.
    if amp_min is not None:
        if "amplitude" not in df.columns:
            print(f"⚠️  No existe columna 'amplitude' en el DF: no puedo filtrar por amp ≥ {amp_min}.")
        else:
            df = df[(df["amplitude"].notna()) & (df["amplitude"] >= amp_min)]

    if df.empty:
        print(f"Sin datos para {station_id} con el filtro de amplitud ≥ {amp_min}.")
        return

    # ---- 2) Grupos por fase (en orden) ----
    group_defs = [("1-8", [1, 8]), ("2-3", [2, 3]), ("4-5", [4, 5]), ("6-7", [6, 7])]

    # Paleta por grupo (sobre negro)
    color_map = {"1-8": "#00E5FF", "2-3": "#FF3DAE", "4-5": "#FFD166", "6-7": "#64FFDA"}
    text_color  = "white"
    face_ax     = "#0d0d0d"
    spine_color = "#A0A0A0"
    grid_color  = "#505050"

    # Extrae datos por grupo (omite grupos vacíos tras el filtro)
    data, labels, colors, Ns = [], [], [], []
    for lab, phases in group_defs:
        vals = df.loc[df["phase"].isin(phases), "N2_s2"].to_numpy()
        if vals.size > 0:
            data.append(vals)
            labels.append(lab)
            colors.append(color_map.get(lab, "#CCCCCC"))
            Ns.append(vals.size)

    if not data:
        print(f"Sin datos en los 4 grupos para {station_id} con amp ≥ {amp_min}.")
        return

    # ---- 3) Figura ----
    fig, ax = plt.subplots(figsize=(10, 5.0))
    fig.patch.set_facecolor("black")
    ax.set_facecolor(face_ax)

    # Halo sutil para destacar líneas sobre fondo oscuro
    glow = [pe.Stroke(linewidth=3.4, foreground=(1, 1, 1, 0.25)), pe.Normal()]

    # Boxplot sin relleno de cajas, con líneas gruesas y outliers visibles
    bp = ax.boxplot(
        data,
        vert=False,
        showfliers=True,
        patch_artist=True,  # permite facecolor='none' y edgecolor por caja
        medianprops=dict(color="white", linewidth=2.4, zorder=3),
        whiskerprops=dict(color="white", linewidth=2.2, zorder=3),
        capprops=dict(color="white", linewidth=2.2, zorder=3),
        flierprops=dict(marker="o", markersize=4.5, markeredgecolor="#CCCCCC",
                        markerfacecolor="#CCCCCC", alpha=0.85, zorder=2),
    )

    # Colores por caja/whiskers/caps
    for i, (box, col) in enumerate(zip(bp["boxes"], colors)):
        box.set(facecolor="none", edgecolor=col, linewidth=2.2, zorder=4)
        box.set_path_effects(glow)

    for i, (w1, w2) in enumerate(zip(bp["whiskers"][::2], bp["whiskers"][1::2])):
        for w in (w1, w2):
            w.set(color=colors[i], linewidth=2.2, zorder=4)
            w.set_path_effects(glow)

    for i, (c1, c2) in enumerate(zip(bp["caps"][::2], bp["caps"][1::2])):
        for c in (c1, c2):
            c.set(color=colors[i], linewidth=2.2, zorder=4)
            c.set_path_effects(glow)

    for m in bp["medians"]:
        m.set_path_effects(glow)

    # --- Etiquetas y escala del eje X en micro (µ s^-2) ---
    yticks = list(range(1, len(labels) + 1))
    ax.set_yticks(yticks)
    ax.set_yticklabels(labels, color=text_color, fontsize=14)

    # Formateo de ticks X: valores*1e6 y sin decimales
    ax.xaxis.set_major_formatter(FuncFormatter(_micro_formatter))
    ax.xaxis.set_major_locator(MaxNLocator(nbins=7))  # ~7 ticks legibles

    # Etiquetas y título
    amp_txt = f" · amp ≥ {amp_min}" if amp_min is not None else ""
    ax.set_xlabel("N² (µ s$^{-2}$)", color=text_color, fontsize=14, labelpad=4)
    ax.set_title(f"{station_id} · N² (980–850 hPa){amp_txt}",
                 color=text_color, fontsize=16, weight="bold", pad=8)

    ax.tick_params(axis="x", colors=text_color, labelsize=14)
    ax.tick_params(axis="y", colors=text_color, labelsize=14)

    # línea de referencia en 0 y cuadrícula sutil
    ax.axvline(0, color="#B0B0B0", lw=1.1, alpha=0.95, zorder=1)
    ax.grid(axis="x", color=grid_color, alpha=0.6, linestyle="-", linewidth=0.9, zorder=0)

    # Marcos
    for spine in ax.spines.values():
        spine.set_color(spine_color)
        spine.set_linewidth(1.0)

    # (opcional) anota n por grupo al lado derecho
    xmax = ax.get_xlim()[1]
    for i, n in enumerate(Ns, start=1):
        ax.text(xmax, i, f"  n={n}", va="center", ha="left", color="#DDDDDD", fontsize=12)

    plt.tight_layout()

    if savepath:
        os.makedirs(os.path.dirname(savepath), exist_ok=True)
        fig.savefig(savepath, bbox_inches="tight", facecolor=fig.get_facecolor())

    plt.show()


# Carpeta para las figuras
FIG_DIR = "./figs"
os.makedirs(FIG_DIR, exist_ok=True)

# Genera un gráfico por estación (amplitud ≥ 1.0 por defecto)
for sid in sorted(daily_2st["station"].unique()):
    horizontal_boxplots_N2(
        daily_2st,
        sid,
        amp_min=1.0,
        savepath=os.path.join(FIG_DIR, f"{sid}_box_N2_dark_nofill_micro_amp1.png")
    )


C:\Users\Camilo\AppData\Local\Temp\ipykernel_4784\1558721097.py:140: UserWarning:

FigureCanvasAgg is non-interactive, and thus cannot be shown



C:\Users\Camilo\AppData\Local\Temp\ipykernel_4784\1558721097.py:140: UserWarning:

FigureCanvasAgg is non-interactive, and thus cannot be shown



#### Celda 16 — Perfiles de θ (rejilla fija), dθ/dz (perfil) y métricas por sondeo

In [18]:
# ============================================================ 
# Celda 16: θ en rejilla fija, dθ/dz (perfil) y métricas por sondeo
#   * N² y θ_low-level: 980–850/900 hPa; gradiente: realmente 0–3 km.
# ============================================================

# Rejilla fija de presión "low-level" (hPa). Forzamos todos los sondeos
# a estos niveles estándar para poder comparar y componer perfiles.
# Se extiende hasta 650 hPa para que el ajuste rotulado 0–3 km alcance esa altura;
# la figura de capa baja continúa mostrando únicamente 980–850 hPa.
PGRID_LOW = np.arange(1000.0, 649.0, -25.0)

# --- Hipsométrica por tramos para obtener alturas (si faltan z observadas) ---
def hypsometric_heights_from_profile(p_hPa, T_K, z0=0.0):
    """
    Integra Δz = (Rd * T̄ / g) * ln(p_i / p_{i+1}) entre niveles adyacentes.
    - p_hPa: presión observada (hPa)
    - T_K  : temperatura en Kelvin
    - z0   : altura de referencia (m)
    Devuelve un array de alturas z(m) con el mismo orden que la entrada.
    """
    p = np.asarray(p_hPa, float)
    T = np.asarray(T_K, float)
    m = np.isfinite(p) & np.isfinite(T)
    if m.sum() < 2:
        return np.full_like(p, np.nan, dtype=float)

    # Orden: superficie (p grande) → tope (p pequeño)
    idx = np.argsort(p[m])[::-1]
    p_sorted = p[m][idx]
    T_sorted = T[m][idx]

    z_sorted = np.zeros_like(p_sorted, dtype=float)
    z_sorted[0] = z0
    for i in range(1, len(p_sorted)):
        Tbar = 0.5*(T_sorted[i-1] + T_sorted[i])
        z_sorted[i] = z_sorted[i-1] + (RD*Tbar/G)*np.log(p_sorted[i-1]/p_sorted[i])

    z_out = np.full_like(p, np.nan, dtype=float)
    z_out[m] = z_sorted[np.argsort(idx)]
    return z_out

def interp_T_theta_on_grid(df_profile, pgrid=PGRID_LOW):
    """
    Interpola T(K) observada a la rejilla pgrid (hPa), calcula θ(K) en esa rejilla
    y deriva zgrid(m) con hipsométrica por tramos usando la T interpolada.
    """
    d = df_profile.dropna(subset=["pressure_hPa", "temp_C"]).copy()
    if d.empty:
        return None

    # Ordenamos por presión descendente (superficie → arriba)
    d = d.sort_values("pressure_hPa", ascending=False)
    p = d["pressure_hPa"].values
    T = d["temp_C"].values + 273.15  # a Kelvin

    # --- Interpolación lineal sin extrapolar en el rango de la rejilla ---
    # (np.interp requiere x creciente; evitamos duplicados en x)
    m = (p >= pgrid.min()) & (p <= pgrid.max())
    if m.sum() < 2:
        Tgrid = np.full_like(pgrid, np.nan, dtype=float)
    else:
        x = p[m][::-1]  # creciente
        y = T[m][::-1]
        x_unique, idx = np.unique(x, return_index=True)
        y_unique = y[idx]
        Tgrid = np.interp(pgrid, x_unique, y_unique, left=np.nan, right=np.nan)

    # θ en la rejilla (Poisson)
    thetagrid = theta_potential_temperature(Tgrid, pgrid)

    # z relativo: se integra desde el mayor nivel de presión válido de la rejilla (z=0).
    zgrid = np.full_like(pgrid, np.nan, dtype=float)
    ok = np.isfinite(Tgrid)
    if ok.sum() >= 2:
        p_ok = pgrid[ok]
        T_ok = Tgrid[ok]
        z_ok = [0.0]
        for i in range(1, len(p_ok)):
            Tbar = 0.5*(T_ok[i-1] + T_ok[i])
            z_ok.append(z_ok[-1] + (RD*Tbar/G)*np.log(p_ok[i-1]/p_ok[i]))
        zgrid[ok] = np.array(z_ok)

    return pgrid.copy(), Tgrid, thetagrid, zgrid

def dtheta_dz_profile(theta_K, z_m):
    """
    Gradiente vertical dθ/dz (K/m) a lo largo del perfil:
    - diferencias centradas en interiores; forward/backward en extremos.
    """
    th = np.asarray(theta_K, float)
    z  = np.asarray(z_m, float)
    ok = np.isfinite(th) & np.isfinite(z)
    dthdz = np.full_like(th, np.nan, dtype=float)
    i = np.where(ok)[0]
    if i.size < 2:
        return dthdz

    for k, ik in enumerate(i):
        if k == 0:
            j = i[k+1]
            dthdz[ik] = (th[j]-th[ik]) / (z[j]-z[ik]) if (z[j]!=z[ik]) else np.nan
        elif k == i.size-1:
            j = i[k-1]
            dthdz[ik] = (th[ik]-th[j]) / (z[ik]-z[j]) if (z[ik]!=z[j]) else np.nan
        else:
            j1, j2 = i[k-1], i[k+1]
            dz = z[j2]-z[j1]
            dth = th[j2]-th[j1]
            dthdz[ik] = dth/dz if dz!=0 else np.nan
    return dthdz  # K/m

def sounding_lowlevel_metrics(df_profile, pgrid=PGRID_LOW):
    """
    Para un sondeo devuelve:
      - N2_s2:    N² 'bulk' 980–850 hPa (s^-2).
      - dthetadz0_3_K_per_km: pendiente lineal de θ entre 0 y 3 km (K/km).
      - theta_ll_K: promedio de θ entre 980 y 900 hPa (K).
      - pgrid, theta_prof_K, dthetadz_prof_K_per_km: vectores para perfiles.
    """
    out = {"N2_s2": np.nan, "dthetadz0_3_K_per_km": np.nan,
           "theta_ll_K": np.nan, "pgrid": pgrid, "theta_prof_K": None,
           "dthetadz_prof_K_per_km": None}

    # N² (bulk) 980–850 hPa
    out["N2_s2"] = bulk_N2_980_850(df_profile)

    # Interpolamos T y θ en la rejilla
    r = interp_T_theta_on_grid(df_profile, pgrid)
    if r is None:
        return out
    pgrid, Tgrid, thetagrid, zgrid = r

    # Guardar perfil θ y su gradiente (K/km)
    out["theta_prof_K"] = thetagrid
    dthdz = dtheta_dz_profile(thetagrid, zgrid) * 1e3
    out["dthetadz_prof_K_per_km"] = dthdz

    # Pendiente lineal θ(z) en 0–3 km (K/km)
    ok = np.isfinite(thetagrid) & np.isfinite(zgrid)
    if ok.sum() >= 2:
        z_ok = zgrid[ok]
        th_ok = thetagrid[ok]
        mwin = (z_ok >= 0.0) & (z_ok <= 3000.0)
        if mwin.sum() >= 2:
            coef = np.polyfit(z_ok[mwin]/1000.0, th_ok[mwin], 1)
            out["dthetadz0_3_K_per_km"] = float(coef[0])

    # θ_low-level: media entre 980 y 900 hPa (antes era 1000–900)
    mll = (pgrid <= 980.0) & (pgrid >= 900.0) & np.isfinite(thetagrid)
    if mll.any():
        out["theta_ll_K"] = float(np.nanmean(thetagrid[mll]))

    return out


#### Celda 17 — Series diarias (métricas + perfiles) para cada estación y unión con RMM

In [19]:
# ============================================================ 
# Celda 17: series diarias (métricas y perfiles) por estación + merge con RMM
# ============================================================
REF_START, REF_END = 1981, 2010  # periodo de climatología DOY (se usa después para calcular anomalías)

def station_daily_metrics_and_profiles(station_id, start="1980-01-01", end=None, pgrid=PGRID_LOW):
    """
    Recorre todos los sondeos de la estación:
      - Calcula métricas por sondeo con sounding_lowlevel_metrics
      - Promedia a diario (si hay 00/12Z el mismo día)
      - Devuelve:
          daily_metrics: [date, theta_ll_K, dthetadz0_3_K_per_km, N2_s2]
          daily_profiles: largo [date, pressure_hPa, theta_K] (promedio diario)
    """
    lines = download_igra_station_file_cached(station_id)  # iterador streaming del TXT dentro del ZIP

    start_ts = pd.Timestamp(start) if start else None      # convierte fechas de recorte a Timestamp (o None)
    end_ts   = pd.Timestamp(end)   if end   else None

    rows_m = []                                          # aquí acumularemos una fila de métricas por sondeo
    profs = []  # guardamos filas en formato largo (date, pressure, theta)  # aquí acumularemos perfiles diarios (θ en pgrid)

    for header, block in iter_soundings(lines):             # procesa y descarta un sondeo a la vez
        hd = parse_header_line(header)
        ts = pd.to_datetime(hd["time"])
        if start_ts and ts < start_ts: 
            continue                                       # salta si está antes del inicio pedido
        if end_ts and ts >= end_ts:
            continue                                       # salta si está al/tras el fin pedido

        level_rows = [parse_level_line(ln) for ln in block]
        if not level_rows:
            continue
        df = pd.DataFrame(level_rows)
        met = sounding_lowlevel_metrics(df, pgrid)         # calcula N²(980–850), dθ/dz(0–3 km) y θ_low-level para este sondeo
        rows_m.append({"station": station_id, "time": ts, "date": ts.floor("D"),  # guarda métricas con la fecha diaria (00Z)
                       "theta_ll_K": met["theta_ll_K"],
                       "dthetadz0_3_K_per_km": met["dthetadz0_3_K_per_km"],
                       "N2_s2": met["N2_s2"]})

        if met["theta_prof_K"] is not None:                # si pudimos interpolar θ en la rejilla de presión…
            for p, th in zip(met["pgrid"], met["theta_prof_K"]):
                profs.append({"station": station_id, "time": ts, "date": ts.floor("D"),
                              "pressure_hPa": float(p), "theta_K": float(th)})  # guardamos el perfil en formato largo

    if not rows_m:                                         # si no hubo datos útiles, devolvemos DFs vacíos con columnas esperadas
        return (pd.DataFrame(columns=["station","date","theta_ll_K","dthetadz0_3_K_per_km","N2_s2",
                                      "rmm1","rmm2","phase","amplitude","group"]),
                pd.DataFrame(columns=["station","date","pressure_hPa","theta_K"]))

    # Promedios diarios: si un día tiene 00Z y 12Z, promediamos (media simple)
    daily_metrics = (pd.DataFrame(rows_m)
                     .groupby(["station","date"], as_index=False)
                     .agg({"theta_ll_K":"mean",
                           "dthetadz0_3_K_per_km":"mean",
                           "N2_s2":"mean"}))

    if profs:                                             # perfiles diarios: media por (estación, fecha, presión)
        daily_profiles = (pd.DataFrame(profs)
                          .groupby(["station","date","pressure_hPa"], as_index=False)
                          .agg({"theta_K":"mean"}))
    else:
        daily_profiles = pd.DataFrame(columns=["station","date","pressure_hPa","theta_K"])

    # Añade RMM por fecha: rmm1, rmm2, fase, amplitud y grupo (1-8, 2-3, 4-5, 6-7)
    daily_metrics = daily_metrics.merge(
        rmm[["date","rmm1","rmm2","phase","amplitude","group"]],
        on="date", how="left"
    )
    return daily_metrics.sort_values(["station","date"]).reset_index(drop=True), \
           daily_profiles.sort_values(["station","date","pressure_hPa"]).reset_index(drop=True)

# Ejecutar para las DOS estaciones elegidas (de la celda de scoring)
sid1, sid2 = chosen["id"].iloc[0], chosen["id"].iloc[1]
metrics_1, profiles_1 = station_daily_metrics_and_profiles(sid1, start="1980-01-01")  # series diarias estación 1
metrics_2, profiles_2 = station_daily_metrics_and_profiles(sid2, start="1980-01-01")  # series diarias estación 2

metrics_all  = pd.concat([metrics_1, metrics_2], ignore_index=True)   # une métricas de ambas estaciones
profiles_all = pd.concat([profiles_1, profiles_2], ignore_index=True) # une perfiles de ambas estaciones

print("metrics_all:", metrics_all.shape, "| profiles_all:", profiles_all.shape)  # tamaños de las tablas resultantes
display(metrics_all.head())       # vista rápida de las primeras filas de métricas + RMM
display(profiles_all.head())      # vista rápida de las primeras filas de perfiles θ(p) diarios


metrics_all: (33118, 10) | profiles_all: (496545, 4)


,station,date,theta_ll_K,dthetadz0_3_K_per_km,N2_s2,rmm1,rmm2,phase,amplitude,group
0,PSM00091408,1980-01-01,302.369615,4.049713,0.000097,0.811586,-0.066468,4.0,0.814303,4-5
1,PSM00091408,1980-01-02,302.722874,4.809806,0.000133,0.705088,-0.029684,4.0,0.705712,4-5
2,PSM00091408,1980-01-03,302.320515,4.614543,0.000112,0.662733,0.037199,5.0,0.663776,4-5
3,PSM00091408,1980-01-04,301.980698,4.559543,0.000121,0.614840,0.213630,5.0,0.650896,4-5
4,PSM00091408,1980-01-05,300.500953,5.510233,0.000168,0.702877,0.490769,5.0,0.857257,4-5


,station,date,pressure_hPa,theta_K
0,PSM00091408,1980-01-01,650.0,NaN
1,PSM00091408,1980-01-01,675.0,315.080807
2,PSM00091408,1980-01-01,700.0,313.880199
3,PSM00091408,1980-01-01,725.0,312.068272
4,PSM00091408,1980-01-01,750.0,310.366798


#### Celda 18 — Climatologías DOY (1981–2010) y anomalías (escalar y perfil)

In [20]:
# ============================================================
# Celda 18: Climatologías DOY (1981–2010) y anomalías (métricas + perfiles)
# ============================================================

def add_doy(df, col="date"):
    # Función auxiliar: añade columnas de año (year) y
    # día-del-año y clave calendario MM-DD a un DataFrame que ya
    # tiene una columna de fechas/tiempos `col`.
    d = df.copy()
    d["year"] = d[col].dt.year         # extrae el año (int)
    d["doy"]  = d[col].dt.dayofyear    # 1..365/366 según fecha
    d["clim_day"] = d[col].dt.strftime("%m-%d")
    # MM-DD evita desplazar marzo–diciembre en años bisiestos; 29-feb usa solo años bisiestos.
    return d

# --- Escalares (θ_ll, dθ/dz_0–3, N²) ---
metrics_all_d = add_doy(metrics_all, "date")  # añade year y doy a la tabla diaria de métricas (ya unida a RMM)
base = metrics_all_d[(metrics_all_d["year"]>=REF_START) & (metrics_all_d["year"]<=REF_END)]
# `base` = subconjunto del período de referencia 1981–2010 (definido en REF_START/REF_END).
# Aquí calcularemos la CLIMATOLOGÍA por día del año (DOY), separada por estación.

clim_metrics = (base.groupby(["station","clim_day"], as_index=False)  # calendario alineado entre años
                     .agg({"theta_ll_K":"mean",                   # promedio climatológico de θ_ll
                           "dthetadz0_3_K_per_km":"mean",         # promedio de (dθ/dz) 0–3 km
                           "N2_s2":"mean"})                       # promedio de N² (980–850 hPa)
                     .rename(columns={"theta_ll_K":"theta_ll_clim",
                                      "dthetadz0_3_K_per_km":"dthetadz0_3_clim",
                                      "N2_s2":"N2_clim"}))        # renombra como *_clim para claridad

metrics_anom = (metrics_all_d
                .merge(clim_metrics, on=["station","clim_day"], how="left"))
metrics_anom["theta_ll_anom"]    = metrics_anom["theta_ll_K"] - metrics_anom["theta_ll_clim"]
# anomalía de θ_ll = valor diario – climatología DOY
metrics_anom["dthetadz0_3_anom"] = metrics_anom["dthetadz0_3_K_per_km"] - metrics_anom["dthetadz0_3_clim"]
# anomalía de (dθ/dz)0–3 km
metrics_anom["N2_anom"]          = metrics_anom["N2_s2"] - metrics_anom["N2_clim"]
# anomalía de N² (980–850 hPa)
# Nota: metrics_anom conserva las columnas de RMM (rmm1, rmm2, phase, amplitude, group)
# que se añadieron en la Celda 17, lo cual permite promediar por fase/grupo.

# --- Perfiles de θ (por presión) ---
profiles_all_d = add_doy(profiles_all, "date")  # añade year/doy a los perfiles diarios θ(p) en formato largo
base_p = profiles_all_d[(profiles_all_d["year"]>=REF_START) & (profiles_all_d["year"]<=REF_END)]
# `base_p` = perfiles del período de referencia para construir la climatología θ_clim(p, DOY)

clim_profiles = (base_p.groupby(["station","clim_day","pressure_hPa"], as_index=False)
                       .agg({"theta_K":"mean"})                 # promedio por estación–DOY–nivel de presión
                       .rename(columns={"theta_K":"theta_clim"}))# renombra a θ_clim para distinguirlo del valor diario

profiles_anom = (profiles_all_d
                 .merge(clim_profiles, on=["station","clim_day","pressure_hPa"], how="left"))
# une a cada perfil diario su perfil climatológico del mismo DOY y nivel de presión

profiles_anom["theta_anom"] = profiles_anom["theta_K"] - profiles_anom["theta_clim"]
# anomalía de perfil: θ'(p) = θ_diario(p) – θ_clim(DOY, p)

print("metrics_anom:", metrics_anom.shape, "| profiles_anom:", profiles_anom.shape)
# tamaños de las tablas resultantes (filas, columnas) para verificación rápida

display(metrics_anom.head())   # muestra ejemplo de las primeras filas con anomalías escalares y columnas RMM
display(profiles_anom.head())  # muestra ejemplo de perfiles con θ, θ_clim y θ_anom por nivel de presión


metrics_anom: (33118, 19) | profiles_anom: (496545, 9)


,station,date,theta_ll_K,dthetadz0_3_K_per_km,N2_s2,rmm1,rmm2,phase,amplitude,group,year,doy,clim_day,theta_ll_clim,dthetadz0_3_clim,N2_clim,theta_ll_anom,dthetadz0_3_anom,N2_anom
0,PSM00091408,1980-01-01,302.369615,4.049713,0.000097,0.811586,-0.066468,4.0,0.814303,4-5,1980,1,01-01,301.413082,4.825293,0.000125,0.956532,-0.775580,-0.000028
1,PSM00091408,1980-01-02,302.722874,4.809806,0.000133,0.705088,-0.029684,4.0,0.705712,4-5,1980,2,01-02,301.428749,4.881618,0.000122,1.294125,-0.071812,0.000011
2,PSM00091408,1980-01-03,302.320515,4.614543,0.000112,0.662733,0.037199,5.0,0.663776,4-5,1980,3,01-03,301.352181,4.900295,0.000124,0.968334,-0.285753,-0.000012
3,PSM00091408,1980-01-04,301.980698,4.559543,0.000121,0.614840,0.213630,5.0,0.650896,4-5,1980,4,01-04,301.270604,5.067855,0.000132,0.710093,-0.508312,-0.000011
4,PSM00091408,1980-01-05,300.500953,5.510233,0.000168,0.702877,0.490769,5.0,0.857257,4-5,1980,5,01-05,301.399944,4.922992,0.000132,-0.898991,0.587242,0.000036


,station,date,pressure_hPa,theta_K,year,doy,clim_day,theta_clim,theta_anom
0,PSM00091408,1980-01-01,650.0,NaN,1980,1,01-01,NaN,NaN
1,PSM00091408,1980-01-01,675.0,315.080807,1980,1,01-01,315.633824,-0.553017
2,PSM00091408,1980-01-01,700.0,313.880199,1980,1,01-01,314.201332,-0.321133
3,PSM00091408,1980-01-01,725.0,312.068272,1980,1,01-01,312.609801,-0.541529
4,PSM00091408,1980-01-01,750.0,310.366798,1980,1,01-01,310.982656,-0.615858


#### Celda 19 — Composites por fase o por grupos 1-8/2-3/4-5/6-7 (con amplitud mínima)

In [21]:
# ============================================================
# Celda 19: composites de anomalías vs RMM (por fase o por grupos)
# ============================================================

def composite_scalar_by_group(df_metrics_anom, station_id, var="N2_anom", amp_min=1.0, by="group"):
    """
    Calcula composiciones (promedios) de una ANOMALÍA ESCALAR por fase/grupo del RMM.

    Parámetros
    ----------
    df_metrics_anom : DataFrame
        Tabla de métricas DIARIAS con anomalías ya calculadas (salida de la Celda 18),
        que además incluye columnas de RMM: 'phase', 'amplitude' y 'group'.
    station_id : str
        ID IGRA de la estación para la cual se hará el composite.
    var : str
        Nombre de la columna con la anomalía escalar que se quiere componer
        (p. ej., "N2_anom", "theta_ll_anom", "dthetadz0_3_anom").
    amp_min : float | None
        Umbral de amplitud del MJO; si se da un número (p. ej., 1.0),
        solo se usan días con MJO "activo" (amplitude >= amp_min).
        Si es None, no se filtra por amplitud.
    by : {"group","phase"}
        Si "group": agrupa en 4 grupos (1-8, 2-3, 4-5, 6-7) como pide el profe.
        Si "phase": agrupa por las 8 fases individuales (1..8).

    Devuelve
    --------
    DataFrame con columnas:
        - key (group o phase)
        - mean   : media de la anomalía en ese grupo/fase
        - stderr : error estándar = std/sqrt(n)
        - n      : número de días usados en el promedio
    """
    # 1) Filtra la estación y asegura que la variable de interés no sea NaN
    d = df_metrics_anom[(df_metrics_anom["station"] == station_id) &
                        (df_metrics_anom[var].notna())].copy()

    # 2) (Opcional) Filtra por amplitud mínima del MJO (actividad >= amp_min)
    if amp_min is not None:
        d = d[(d["amplitude"].notna()) & (d["amplitude"] >= amp_min)]

    # 3) Selector de clave de agrupación: 4 grupos ("group") o fases 1..8 ("phase")
    key = "group" if by == "group" else "phase"

    # 4) Agrupa por clave y calcula:
    #    - mean: promedio de la anomalía
    #    - std : desviación estándar muestral (ddof=1)
    #    - n   : tamaño de muestra
    #    Se hace en un solo .agg para mantener alineados los índices/filas.
    stats = (d.groupby(key)[var]
               .agg(mean="mean",
                    std=lambda x: x.std(ddof=1),
                    n="size")
               .reset_index())

    # 5) Error estándar de la media: std / sqrt(n). Protege n=0 con clip(lower=1).
    stats["stderr"] = stats["std"] / np.sqrt(stats["n"].clip(lower=1))

    # 6) Orden “bonito” de salida 
    if by == "group":
        order = ["1-8", "2-3", "4-5", "6-7"]
        stats[key] = pd.Categorical(stats[key], categories=order, ordered=True)
        stats = stats.sort_values(key)
    else:
        stats = stats.sort_values(key)  # fases 1..8 quedan en orden ascendente

    # 7) Devuelve solo las columnas necesarias, con índice limpio
    return stats[[key, "mean", "stderr", "n"]].reset_index(drop=True)


def composite_profile_by_group(df_profiles_anom, station_id, amp_min=1.0, by="group"):
    """
    Calcula COMPOSITES DE PERFILES de anomalía de θ (θ_anom) por grupo/fase del RMM.

    Parámetros
    ----------
    df_profiles_anom : DataFrame
        Tabla en formato "largo" con perfiles diarios: columnas
        ['station','date','pressure_hPa','theta_K','theta_clim','theta_anom', 'year','doy'].
        (Salida de la Celda 18 para perfiles).
    station_id : str
        ID IGRA de la estación.
    amp_min : float | None
        Umbral de amplitud del MJO para filtrar días “activos” (igual que arriba).
    by : {"group","phase"}
        Agrupar por 4 grupos o por 8 fases.

    Devuelve
    --------
    dict[group/fase -> DataFrame] donde cada DataFrame tiene columnas:
        - pressure_hPa
        - theta_anom_mean : promedio de la anomalía de θ en ese nivel
        - n               : número de días que entran en ese promedio
    """
    # 1) Filtra la estación y exige θ_anom válido (si no, no se puede promediar)
    d = df_profiles_anom[(df_profiles_anom["station"]==station_id) &
                         (df_profiles_anom["theta_anom"].notna())].copy()

    # 2) Añade columnas de RMM para cada fecha del perfil.
    #    IMPORTANTE: aquí se usa metrics_all (no la versión con anomalías) porque contiene
    #    la información de RMM por 'date' que necesitamos (phase, amplitude y group).
    d = d.merge(metrics_all[["station","date","phase","amplitude","group"]],
                on=["station","date"], how="left")

    # 3) (Opcional) Filtra por amplitud mínima (MJO activo)
    if amp_min is not None:
        d = d[(d["amplitude"].notna()) & (d["amplitude"]>=amp_min)]

    # 4) Define la clave de agrupación (4 grupos o fases) para el composite
    key = "group" if by=="group" else "phase"

    # 5) Para cada grupo/fase, promedia la ANOMALÍA de θ por nivel de presión:
    #    mean de θ_anom y n de días que contribuyen en ese nivel.
    comp = {}
    for g, gg in d.groupby(key):
        agg = (gg.groupby("pressure_hPa")["theta_anom"]
                 .agg(["mean","size"]).reset_index()
                 .rename(columns={"size":"n","mean":"theta_anom_mean"}))
        # 6) Ordena el perfil de mayor a menor presión (superficie → arriba) para graficar como T–p
        comp[g] = agg.sort_values("pressure_hPa", ascending=False).reset_index(drop=True)

    # 7) Devuelve un diccionario: cada clave (p. ej., "1-8") mapea a su perfil compuesto
    return comp


#### Celda 20 — Gráficas de composites (perfil de θ y barras para escalares)

In [22]:
# ============================================================ 
# Celda 20 (oscura): perfiles θ′ y figura unificada con 3 barras
# ============================================================
import os
import numpy as np
import matplotlib.pyplot as plt

# ---- ajustes globales útiles para fondo negro
plt.rcParams.update({
    "figure.facecolor": "#000000",
    "axes.facecolor":   "#000000",
    "savefig.facecolor":"#000000",
    "axes.edgecolor":   "#AAAAAA",
    "grid.color":       "#777777",
})

FIG_DIR = "./figs"
os.makedirs(FIG_DIR, exist_ok=True)

# Paleta que resalta sobre negro (una por grupo)
GROUP_COLORS = {
    "1-8": "#00D1FF",  # cian brillante
    "2-3": "#FFB000",  # naranja
    "4-5": "#3DFA8E",  # verde lima
    "6-7": "#FF4D6D",  # rojo/rosa
}

# ---------- utilidades de estilo ----------
def _style_axes(ax, title=None, xlabel=None, ylabel=None, title_weight="bold"):
    """Aplica colores y tamaños de fuente para tema oscuro."""
    if title is not None:
        ax.set_title(title, color="white", fontsize=16, fontweight=title_weight)
    if xlabel is not None:
        ax.set_xlabel(xlabel, color="white", fontsize=14)
    if ylabel is not None:
        ax.set_ylabel(ylabel, color="white", fontsize=14)

    ax.tick_params(colors="white", labelsize=14)
    for spine in ax.spines.values():
        spine.set_color("#AAAAAA")
    ax.grid(True, alpha=0.35)

def _legend_outside(ax, title=""):
    """Leyenda fuera del área del eje, a la derecha."""
    leg = ax.legend(
        title=title, loc="upper left", bbox_to_anchor=(1.02, 1.0),
        frameon=True, borderaxespad=0.0
    )
    # Colores de leyenda para tema oscuro
    leg.get_frame().set_facecolor("#222222")
    leg.get_frame().set_edgecolor("#555555")
    if leg.get_title():
        leg.get_title().set_color("white")
    for txt in leg.get_texts():
        txt.set_color("white")

# ---------- figura 1: perfiles θ′ vs presión ----------
def plot_composite_theta_profiles(
    station_id, comp_dict, title_suffix="", by="group",
    savepath=None, dpi=300
):
    order = ["1-8","2-3","4-5","6-7"] if by=="group" else list(range(1,9))
    labels = [str(x) for x in order]

    fig, ax = plt.subplots(figsize=(14, 6), dpi=dpi)

    for lab in labels:
        if lab in comp_dict:
            dfp = comp_dict[lab]  # columns: pressure_hPa, theta_anom_mean, n
            ax.plot(
                dfp["theta_anom_mean"], dfp["pressure_hPa"],
                marker="o", ms=6, mec="white", mew=0.7,
                lw=2.2, color=GROUP_COLORS.get(lab, "#bbbbbb"),
                label=lab
            )

    ax.invert_yaxis()
    ax.set_ylim(980, 850)   # ← AQUÍ: recorte visual a 980–850 hPa

    _style_axes(
        ax,
        title=f"{station_id} · Composite θ' vs RMM {title_suffix}",
        xlabel="θ anomalía (K)",
        ylabel="Presión (hPa)",
    )
    _legend_outside(ax, title=by)

    # deja margen a la derecha para la leyenda
    fig.tight_layout(rect=[0.0, 0.0, 0.80, 1.0])

    if savepath:
        fig.savefig(savepath, bbox_inches="tight")
    plt.show()


# ---------- figura 2: 3 subplots de barras unificados ----------
def _bar_on_ax(ax, df, key, label_y):
    """Barras con error y línea 0, coloreadas por grupo."""
    x = df[key].astype(str).values
    y = df["mean"].values
    e = df["stderr"].values
    colors = [GROUP_COLORS.get(xx, "#888888") for xx in x]

    ax.bar(x, y, yerr=e, capsize=5,
           linewidth=1.2, edgecolor="#E6E6E6", color=colors)
    ax.axhline(0, lw=1.2, color="#AAAAAA")
    _style_axes(ax, xlabel=("Grupo RMM" if key == "group" else "Fase RMM"), ylabel=label_y)

def plot_unified_scalar_composites(
    station_id, comp_theta_ll, comp_dtdz, comp_N2,
    by="group", savepath=None, dpi=300
):
    key = "group" if by=="group" else "phase"
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=dpi)

    _bar_on_ax(axes[0], comp_theta_ll, key, r"$\theta$ low-level$^\prime$ (K)")
    _bar_on_ax(axes[1], comp_dtdz,     key, r"(d$\theta$/dz)$^\prime$ 0–3 km (K km$^{-1}$)")
    _bar_on_ax(axes[2], comp_N2,       key, r"$N^{2\prime}$ (s$^{-2}$)")

    # título general
    fig.suptitle(f"{station_id} · Composite de anomalías",
                 color="white", fontsize=16, y=1.03, fontweight="bold")
    fig.tight_layout()

    if savepath:
        fig.savefig(savepath, bbox_inches="tight")
    plt.show()


# -------------------- Ejecutar por estación --------------------
AMP_MIN = 1.0  # umbral de MJO activo

for sid in metrics_anom["station"].unique():
    # 1) Perfiles de θ′ por grupo
    comp_theta = composite_profile_by_group(profiles_anom, sid, amp_min=AMP_MIN, by="group")
    plot_composite_theta_profiles(
        sid, comp_theta, title_suffix=f"(amp ≥ {AMP_MIN})", by="group",
        savepath=os.path.join(FIG_DIR, f"{sid}_theta_profile_dark.png")
    )

    # 2) Figura unificada con las 3 barras
    comp_theta_ll = composite_scalar_by_group(metrics_anom, sid, var="theta_ll_anom",     amp_min=AMP_MIN, by="group")
    comp_dtdz     = composite_scalar_by_group(metrics_anom, sid, var="dthetadz0_3_anom", amp_min=AMP_MIN, by="group")
    comp_N2       = composite_scalar_by_group(metrics_anom, sid, var="N2_anom",           amp_min=AMP_MIN, by="group")

    plot_unified_scalar_composites(
        sid, comp_theta_ll, comp_dtdz, comp_N2, by="group",
        savepath=os.path.join(FIG_DIR, f"{sid}_composites_barras_dark.png")
    )


C:\Users\Camilo\AppData\Local\Temp\ipykernel_4784\1275920308.py:93: UserWarning:

FigureCanvasAgg is non-interactive, and thus cannot be shown



C:\Users\Camilo\AppData\Local\Temp\ipykernel_4784\1275920308.py:127: UserWarning:

FigureCanvasAgg is non-interactive, and thus cannot be shown



C:\Users\Camilo\AppData\Local\Temp\ipykernel_4784\1275920308.py:93: UserWarning:

FigureCanvasAgg is non-interactive, and thus cannot be shown



C:\Users\Camilo\AppData\Local\Temp\ipykernel_4784\1275920308.py:127: UserWarning:

FigureCanvasAgg is non-interactive, and thus cannot be shown



##### Qué esperar al correr

##### Nuevas tablas:

* metrics_all (diaria) y profiles_all (θ por presión).

* metrics_anom y profiles_anom (anomalías DOY 81–10).

##### Figuras nuevas:

* Composite de θ’(p) por grupos RMM (4 líneas).

* Barras con media ± stderr de θ_low-level’, (dθ/dz)_{0–3 km}’ y N²’ por grupos.

##### Sigues teniendo tus histogramas por fase y boxplots por grupos de N², que eran parte de los entregables.

##### Si quieres que, además de grupos, saque fases 1..8 en los composites, cambia by="group" por by="phase" en las llamadas de la Celda 20 (y listo).

In [23]:
# ============================================================
# Celda 21: Visualización del índice RMM (fase-espacio) — DARK
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def _get_rmm_df_for_plot():
    """
    Devuelve DataFrame indexado por fecha con columnas ['rmm1','rmm2','phase','amplitude'].
    Usa 'rmm' ya cargado; si no existe, intenta cargar desde RMM_LOCAL_PATH.
    """
    try:
        df = rmm.copy()
        if "date" in df.columns:
            df = df.set_index("date")
        return df[["rmm1","rmm2","phase","amplitude"]].dropna().sort_index()
    except NameError:
        df = load_rmm_from_file(RMM_LOCAL_PATH)
        return df[["rmm1","rmm2","phase","amplitude"]].dropna().sort_index()

def plot_rmm_phase_space_matplotlib(df, highlight_days=60, label_every_days=5, savepath=None):
    """
    Diagrama RMM (RMM2 vs RMM1) con tema oscuro:
      - trayectoria completa en gris
      - últimos 'highlight_days' con color (Viridis)
      - círculo de amplitud = 1
      - ejes, diagonales y cuadrícula sutil
      - números de fase y nombres de regiones
    """
    df = df.dropna(subset=["rmm1","rmm2"]).sort_index()

    # --- Colores / estilo dark ---
    face = "#000000"
    gridc = (1, 1, 1, 0.18)
    linec = "#B0B0B0"
    textc = "white"

    fig, ax = plt.subplots(figsize=(11, 12), dpi=150)
    fig.patch.set_facecolor(face)
    ax.set_facecolor(face)

    # Trayectoria completa (tenue)
    ax.plot(df["rmm1"].values, df["rmm2"].values, lw=0.9, color="#8c8c8c", alpha=0.55, label="Trayectoria completa")

    # Segmento reciente coloreado
    cutoff = df.index.max() - pd.Timedelta(days=highlight_days)
    recent = df[df.index >= cutoff]
    if len(recent) >= 2:
        t = (recent.index - recent.index.min()).days.values.astype(float)
        sc = ax.scatter(
            recent["rmm1"], recent["rmm2"],
            c=t, cmap="viridis", s=28, zorder=3,
            edgecolors="white", linewidths=0.3
        )
        ax.plot(recent["rmm1"], recent["rmm2"], lw=2.0, color="#00D1FF", alpha=0.95,
                label=f"Últimos {highlight_days} días")

        # Colorbar legible en oscuro
        cbar = fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.02)
        cbar.set_label("Días desde el inicio", color=textc, fontsize=12)
        cbar.ax.tick_params(colors=textc, labelsize=12)
        cbar.outline.set_edgecolor(textc)

        # Etiquetas de fecha (opcionales)
        if label_every_days and highlight_days:
            step = max(1, int(round(len(recent) / (highlight_days / label_every_days))))
            for i, (dt, row) in enumerate(recent.iterrows()):
                if i % step == 0:
                    ax.text(row["rmm1"], row["rmm2"], dt.strftime("%d%b").lower(),
                            fontsize=10, color="white", ha="left", va="bottom",
                            bbox=dict(boxstyle="round,pad=0.15", fc=(0,0,0,0.35), ec="none"))

    # Último punto en rojo (con borde blanco)
    last = df.iloc[-1]
    ax.plot(last["rmm1"], last["rmm2"], 'o', ms=9, color="#FF4D6D",
            mec="white", mew=1.5, zorder=5,
            label=f"Último: {df.index[-1].date()}  (fase {int(last['phase'])}, amp {last['amplitude']:.2f})")

    # Círculo de amplitud = 1
    th = np.linspace(0, 2*np.pi, 361)
    ax.plot(np.cos(th), np.sin(th), ls=(0, (5, 3)), color=linec, lw=1.2, label="Amplitud = 1")

    # Ejes, diagonales y cuadrícula
    ax.axhline(0, color=gridc, lw=1.1)
    ax.axvline(0, color=gridc, lw=1.1)
    ax.plot([-4, 4], [-4, 4], ls=":", color=gridc, lw=0.9)
    ax.plot([-4, 4], [ 4,-4], ls=":", color=gridc, lw=0.9)
    ax.set_xlim(-4, 4); ax.set_ylim(-4, 4)
    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, color=gridc, linewidth=0.9, alpha=0.5)

    # Números de fase
    phase_pos = {
        1:(-3.0,-2.0), 2:(-2.0,-3.0), 3:( 2.0,-3.0), 4:( 3.0,-2.0),
        5:( 3.0, 2.0), 6:( 2.0, 3.0), 7:(-2.0, 3.0), 8:(-3.0, 2.0),
    }
    for ph, (x, y) in phase_pos.items():
        ax.text(x, y, str(ph), ha="center", va="center",
                fontsize=12, color="white", fontweight="bold")

    # Etiquetas de regiones
    ax.text( 0.0,  3.35, "Western\nPacific",   ha="center", va="center", fontsize=12, color="#DDDDDD")
    ax.text( 3.35, 0.00, "Maritime\nContinent",ha="center", va="center", fontsize=12, color="#DDDDDD", rotation=270)
    ax.text( 0.0, -3.35, "Indian\nOcean",      ha="center", va="center", fontsize=12, color="#DDDDDD")
    ax.text(-3.35, 0.00, "West. Hem.\n& Africa",ha="center", va="center", fontsize=12, color="#DDDDDD", rotation=90)

    # Ejes y título en blanco (16/14)
    ax.set_xlabel("RMM1", color=textc, fontsize=12)
    ax.set_ylabel("RMM2", color=textc, fontsize=12)
    ax.set_title("RMM Phase Space (RMM2 vs RMM1)", color=textc, fontsize=14, fontweight="bold")
    ax.tick_params(colors=textc, labelsize=12)
    for s in ax.spines.values():  # bordes del eje
        s.set_color("#AAAAAA")

    # Leyenda fuera de la gráfica
    leg = ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1.0), frameon=True)
    leg.get_frame().set_facecolor("#222222")
    leg.get_frame().set_edgecolor("#555555")
    if leg.get_title(): leg.get_title().set_color("white")
    for t in leg.get_texts(): t.set_color("white")

    # deja margen a la derecha para la leyenda
    fig.tight_layout(rect=[0.0, 0.0, 0.86, 1.0])

    if savepath:
        import os
        os.makedirs(os.path.dirname(savepath), exist_ok=True)
        fig.savefig(savepath, dpi=300, bbox_inches="tight", facecolor=fig.get_facecolor())

    plt.show()
    return fig, ax

# Ejecutar la visualización (ajusta highlight_days si quieres)
_rmm_df = _get_rmm_df_for_plot()
plot_rmm_phase_space_matplotlib(_rmm_df, highlight_days=60, label_every_days=5, savepath=None)


C:\Users\Camilo\AppData\Local\Temp\ipykernel_4784\4090023161.py:131: UserWarning:

FigureCanvasAgg is non-interactive, and thus cannot be shown



(<Figure size 1650x1800 with 2 Axes>,
 <Axes: title={'center': 'RMM Phase Space (RMM2 vs RMM1)'}, xlabel='RMM1', ylabel='RMM2'>)

In [24]:
# ============================================================
# GIF Pre-acondicionamiento MJO–N² (980–850 hPa) — versión mejorada
# - Panel del mapa (verde/marrón) MÁS ancho
# - Leyenda fuera del mapa (no tapa el campo)
# - Panel de N² por estación: barras claras, números grandes, colorbar fija
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation, colors
import warnings

# (opcional) cartopy para dibujar continentes
_HAS_CARTOPY = False
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    _HAS_CARTOPY = True
except Exception:
    warnings.warn("cartopy no disponible: se usará un fondo simple.")

# ======= ESTILO GENERAL =======
plt.rcParams.update({
    "figure.facecolor": "black",
    "axes.facecolor":   "black",
    "savefig.facecolor":"black",
    "text.color":       "white",
    "axes.edgecolor":   "#AAAAAA",
    "axes.labelcolor":  "white",
    "xtick.color":      "white",
    "ytick.color":      "white",
    "font.size":        14,
})

# ======= Fases → región y longitudes de centro (ºE) =======
PHASE_INFO = {
    1: {"region": "Western Hemisphere & Africa", "lon_c": -30},
    2: {"region": "Indian Ocean (W)",            "lon_c":  60},
    3: {"region": "Indian Ocean (E)",            "lon_c":  90},
    4: {"region": "Maritime Continent (W)",      "lon_c": 120},
    5: {"region": "Maritime Continent (E)",      "lon_c": 140},
    6: {"region": "Western Pacific",             "lon_c": 160},
    7: {"region": "Central/E Pacific",           "lon_c": -140},
    8: {"region": "Africa/Atlantic (return)",    "lon_c":  -10},
}
PHASE_ORDER = [1,2,3,4,5,6,7,8]

# Paletas
CMAP_STAB = plt.cm.coolwarm  # estabilidad (N²)
GRID_COL  = "#666666"

# ======= Prepara N² por fase (μ s^-2) =======
def _prep_phase_stats(master_df, amp_min=1.0):
    d = master_df.copy()
    d = d[(d["N2_s2"].notna()) & (d["phase"].notna())]
    if amp_min is not None:
        d = d[(d["amplitude"].notna()) & (d["amplitude"] >= amp_min)]
    if d.empty:
        raise ValueError("No hay datos válidos de N² con el filtro actual.")
    g = (d.groupby(["station","phase"])["N2_s2"]
           .mean()
           .mul(1e6)               # a μ s^-2
           .rename("N2_us2"))
    vmin, vmax = float(g.min()), float(g.max())
    stations = sorted(d["station"].unique())
    return g, vmin, vmax, stations

# ======= Campos sintéticos de fase (verde/marrón) =======
def _phase_fields(lon_c, sigma=35.0, lat_half=18.0):
    lons = np.linspace(-180, 180, 721)
    lats = np.linspace(-30, 30, 241)
    LON, LAT = np.meshgrid(lons, lats)
    def wrapdist(a, b):
        d = (a - b + 180) % 360 - 180
        return d
    gauss_lon = np.exp(-0.5*(wrapdist(LON, lon_c)/sigma)**2)
    gauss_lat = np.exp(-0.5*((LAT/lat_half))**2)
    conv = gauss_lon * gauss_lat
    supp = np.exp(-0.5*(wrapdist(LON, (lon_c+180)) / sigma)**2) * gauss_lat
    return LON, LAT, conv, supp

# ======= Fondo del mapa =======
def _add_map_background(ax):
    if _HAS_CARTOPY:
        ax.set_global()
        ax.set_extent([-180, 180, -30, 30], crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.LAND.with_scale("110m"), facecolor="#2b2b2b", edgecolor="#404040", linewidth=0.4)
        ax.coastlines(color="#9A9A9A", linewidth=0.7)
        ax.gridlines(draw_labels=False, color=GRID_COL, linewidth=0.4, linestyle=":")
    else:
        ax.set_xlim(-180, 180); ax.set_ylim(-30, 30)
        for lon in range(-180, 181, 30):
            ax.plot([lon, lon], [-30, 30], color=GRID_COL, lw=0.4, ls=":")
        for lat in range(-30, 31, 10):
            ax.plot([-180, 180], [lat, lat], color=GRID_COL, lw=0.4, ls=":")
        ax.fill_between([-180, 180], -30, 30, color="#222222", alpha=0.35)
        ax.set_xlabel("Longitud (°)"); ax.set_ylabel("Latitud (°)")

# ======= Panel de barras N² (claro y con colorbar) =======
def _draw_stability_panel(ax, stations, phase, stats_series, norm, cmap, vmax):
    ax.clear()
    ax.set_facecolor("black")
    for sp in ax.spines.values():
        sp.set_color("#AAAAAA")
    # valores por estación en esta fase
    vals = [stats_series.get((st, phase), np.nan) for st in stations]
    y = np.arange(len(stations))[::-1]
    cols = [cmap(norm(v)) if np.isfinite(v) else (0.4,0.4,0.4,0.3) for v in vals]
    ax.barh(y, [0 if not np.isfinite(v) else v for v in vals],
            color=cols, edgecolor="#E6E6E6", linewidth=1.2, height=0.55)
    # Números grandes al final de cada barra
    for yy, v in zip(y, vals):
        txt = "s/d" if not np.isfinite(v) else f"{v:,.0f}"
        xend = 0 if not np.isfinite(v) else v
        ax.text(xend + vmax*0.02, yy, txt, va="center", fontsize=18, fontweight="bold", color="white")
    ax.set_yticks(y); ax.set_yticklabels(stations, fontsize=15)
    ax.set_xlabel("N² (μ s$^{-2}$)", fontsize=15, labelpad=6)
    ax.set_xlim(0, vmax*1.18)
    ax.grid(axis="x", color="#555555", alpha=0.5, lw=0.7)
    ax.set_title(f"Fase {phase}: N² medio", fontsize=13, pad=10)

# ======= Generador del GIF =======
def make_mjo_preconditioning_gif(master_df, amp_min=1.0, outfile="mjo_precond.gif", dpi=150):
    stats, vmin, vmax, stations = _prep_phase_stats(master_df, amp_min=amp_min)
    norm = colors.Normalize(vmin=vmin, vmax=vmax)
    cmap = CMAP_STAB

    # --- figura más ancha para el mapa ---
    fig = plt.figure(figsize=(16, 6.2), dpi=dpi)
    gs = fig.add_gridspec(1, 2, width_ratios=[3.3, 1.5], wspace=0.35)

    # MAPA
    if _HAS_CARTOPY:
        ax_map = fig.add_subplot(gs[0,0], projection=ccrs.PlateCarree())
    else:
        ax_map = fig.add_subplot(gs[0,0])
    _add_map_background(ax_map)

    # BARRAS
    ax_bar = fig.add_subplot(gs[0,1])

    # Leyenda fuera del mapa (no tapa el campo)
    fig.text(0.04, 0.90, "Envolvente MJO esquemática\nConvección (verde) · suprimida (marrón)",
             ha="left", va="top",
             fontsize=13, color="white",
             bbox=dict(facecolor="#0f0f0f", edgecolor="#4d4d4d", boxstyle="round,pad=0.35"))

    # Colormesh de conv/supp (placeholders)
    lon_init = PHASE_INFO[1]["lon_c"]
    LON, LAT, conv0, supp0 = _phase_fields(lon_init)
    if _HAS_CARTOPY:
        im1 = ax_map.pcolormesh(LON, LAT, conv0, transform=ccrs.PlateCarree(), shading="auto", cmap="Greens", vmin=0, vmax=1)
        im2 = ax_map.pcolormesh(LON, LAT, supp0, transform=ccrs.PlateCarree(), shading="auto", cmap="BrBG_r", vmin=0, vmax=1)
    else:
        im1 = ax_map.pcolormesh(LON, LAT, conv0, shading="auto", cmap="Greens", vmin=0, vmax=1)
        im2 = ax_map.pcolormesh(LON, LAT, supp0, shading="auto", cmap="BrBG_r", vmin=0, vmax=1)
    im1.set_alpha(0.55); im2.set_alpha(0.45)

    # Título dinámico
    supt = fig.suptitle("", fontsize=24, fontweight="bold", color="white", y=0.98)

    # Colorbar fija para N² (debajo de las barras)
    sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
    cbar = fig.colorbar(sm, ax=ax_bar, orientation="horizontal", fraction=0.12, pad=0.22)
    cbar.set_label("N² (μ s$^{-2}$) – estabilidad", fontsize=13)
    cbar.ax.tick_params(labelsize=11)

    # Primera pinta del panel de barras
    _draw_stability_panel(ax_bar, stations, PHASE_ORDER[0], stats, norm, cmap, vmax)

    # Función de actualización por fase
    def _update(k):
        phase = PHASE_ORDER[k]
        info = PHASE_INFO[phase]
        lon_c = info["lon_c"]
        _, _, conv_k, supp_k = _phase_fields(lon_c)
        im1.set_array(conv_k.ravel())
        im2.set_array(supp_k.ravel())
        _draw_stability_panel(ax_bar, stations, phase, stats, norm, cmap, vmax)
        supt.set_text(f"Pre-acondicionamiento MJO · Fase {phase} — {info['region']}   (amp ≥ {amp_min})")
        return (im1, im2, supt)

    ani = animation.FuncAnimation(
    fig, _update,
    frames=len(PHASE_ORDER),
    interval=10000,        # <- antes 1200. Ahora 2.8 s por fase
    repeat=True,
    repeat_delay=5000     # <- pausa de 2.0 s al terminar la vuelta
)


    try:
        ani.save(outfile, writer="pillow", dpi=dpi)
        print(f"✅ GIF generado: {outfile}")
    except Exception as e:
        print("No pude guardar GIF (¿falta pillow?). Exporto PNGs:")
        for i in range(len(PHASE_ORDER)):
            _update(i)
            fn = outfile.replace(".gif","") + f"_phase{i+1}.png"
            plt.savefig(fn, bbox_inches="tight", dpi=dpi)
            print("  •", fn)
    plt.close(fig)

# ===== Ejecutar =====
# Cambia amp_min=None si no quieres filtrar por MJO activo
make_mjo_preconditioning_gif(daily_2st, amp_min=1.0, outfile="mjo_precond.gif", dpi=160)


C:\Users\Camilo\AppData\Local\Programs\Python\Python311\Lib\site-packages\cartopy\io\__init__.py:242: DownloadWarning:

Downloading: https://naturalearth.s3.amazonaws.com/110m_physical/ne_110m_land.zip



C:\Users\Camilo\AppData\Local\Programs\Python\Python311\Lib\site-packages\cartopy\io\__init__.py:242: DownloadWarning:

Downloading: https://naturalearth.s3.amazonaws.com/110m_physical/ne_110m_coastline.zip



✅ GIF generado: mjo_precond.gif


In [25]:
# ============================================================
# MJO pre-acondicionamiento · 8 IMÁGENES (una por fase)
# - Misma estética que el GIF mejorado
# - Mapa convectivo (verde) / suprimido (marrón) más ANCHO
# - Panel de N² por estación con colorbar FIJA y etiquetas grandes
# - Exporta PNGs (y opcionalmente un PDF multipágina)
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors

# ---------- (opcional) cartopy para costas ----------
_HAS_CARTOPY = False
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    _HAS_CARTOPY = True
except Exception:
    warnings.warn("cartopy no disponible: se usará un fondo simple.")

# ---------- estilo general (tema oscuro) ----------
plt.rcParams.update({
    "figure.facecolor": "black",
    "axes.facecolor":   "black",
    "savefig.facecolor":"black",
    "text.color":       "white",
    "axes.edgecolor":   "#AAAAAA",
    "axes.labelcolor":  "white",
    "xtick.color":      "white",
    "ytick.color":      "white",
    "font.size":        14,
})

# ---------- Fases → región y longitudes de centro (ºE) ----------
PHASE_INFO = {
    1: {"region": "Western Hemisphere & Africa", "lon_c": -30},
    2: {"region": "Indian Ocean (W)",            "lon_c":  60},
    3: {"region": "Indian Ocean (E)",            "lon_c":  90},
    4: {"region": "Maritime Continent (W)",      "lon_c": 120},
    5: {"region": "Maritime Continent (E)",      "lon_c": 140},
    6: {"region": "Western Pacific",             "lon_c": 160},
    7: {"region": "Central/E Pacific",           "lon_c": -140},
    8: {"region": "Africa/Atlantic (return)",    "lon_c":  -10},
}
PHASE_ORDER = [1,2,3,4,5,6,7,8]

# ---------- colormap para estabilidad ----------
CMAP_STAB = plt.cm.coolwarm
GRID_COL   = "#666666"

# ---------- N² por fase (μ s^-2) con misma escala para TODAS las fases ----------
def _prep_phase_stats(master_df, amp_min=1.0):
    d = master_df.copy()
    d = d[(d["N2_s2"].notna()) & (d["phase"].notna())]
    if amp_min is not None:
        d = d[(d["amplitude"].notna()) & (d["amplitude"] >= amp_min)]
    if d.empty:
        raise ValueError("No hay datos válidos de N² con el filtro actual.")
    g = (d.groupby(["station","phase"])["N2_s2"]
           .mean()
           .mul(1e6)               # a μ s^-2
           .rename("N2_us2"))
    vmin, vmax = float(g.min()), float(g.max())
    stations = sorted(d["station"].unique())
    return g, vmin, vmax, stations

# ---------- campo sintético convectivo/suprimido por fase ----------
def _phase_fields(lon_c, sigma=35.0, lat_half=18.0):
    lons = np.linspace(-180, 180, 721)
    lats = np.linspace(-30, 30, 241)
    LON, LAT = np.meshgrid(lons, lats)
    def wrapdist(a, b):
        return (a - b + 180) % 360 - 180
    gauss_lon = np.exp(-0.5*(wrapdist(LON, lon_c)/sigma)**2)
    gauss_lat = np.exp(-0.5*((LAT/lat_half))**2)
    conv = gauss_lon * gauss_lat
    supp = np.exp(-0.5*(wrapdist(LON, (lon_c+180))/sigma)**2) * gauss_lat
    return LON, LAT, conv, supp

# ---------- fondo del mapa ----------
def _add_map_background(ax):
    if _HAS_CARTOPY:
        ax.set_global()
        ax.set_extent([-180, 180, -30, 30], crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.LAND.with_scale("110m"),
                       facecolor="#2b2b2b", edgecolor="#404040", linewidth=0.4)
        ax.coastlines(color="#9A9A9A", linewidth=0.7)
        ax.gridlines(draw_labels=False, color=GRID_COL, linewidth=0.4, linestyle=":")
    else:
        ax.set_xlim(-180, 180); ax.set_ylim(-30, 30)
        for lon in range(-180, 181, 30):
            ax.plot([lon, lon], [-30, 30], color=GRID_COL, lw=0.4, ls=":")
        for lat in range(-30, 31, 10):
            ax.plot([-180, 180], [lat, lat], color=GRID_COL, lw=0.4, ls=":")
        ax.fill_between([-180, 180], -30, 30, color="#222222", alpha=0.35)
        ax.set_xlabel("Longitud (°)"); ax.set_ylabel("Latitud (°)")

# ---------- panel de barras N² ----------
def _draw_stability_panel(ax, stations, phase, stats_series, norm, cmap, vmax):
    ax.clear()
    ax.set_facecolor("black")
    for sp in ax.spines.values():
        sp.set_color("#AAAAAA")
    vals = [stats_series.get((st, phase), np.nan) for st in stations]
    y = np.arange(len(stations))[::-1]
    cols = [cmap(norm(v)) if np.isfinite(v) else (0.4,0.4,0.4,0.3) for v in vals]
    ax.barh(y, [0 if not np.isfinite(v) else v for v in vals],
            color=cols, edgecolor="#E6E6E6", linewidth=1.2, height=0.55)
    for yy, v in zip(y, vals):
        txt = "s/d" if not np.isfinite(v) else f"{v:,.0f}"
        xend = 0 if not np.isfinite(v) else v
        ax.text(xend + vmax*0.02, yy, txt, va="center", fontsize=18,
                fontweight="bold", color="white")
    ax.set_yticks(y); ax.set_yticklabels(stations, fontsize=15)
    ax.set_xlabel("N² (μ s$^{-2}$)", fontsize=15, labelpad=6)
    ax.set_xlim(0, vmax*1.18)
    ax.grid(axis="x", color="#555555", alpha=0.5, lw=0.7)
    ax.set_title(f"Fase {phase}: N² medio", fontsize=13, pad=10)

# ---------- función principal: guarda 8 PNG (y PDF opcional) ----------
def save_mjo_preconditioning_phases(master_df,
                                    amp_min=1.0,
                                    outdir="mjo_phases",
                                    basename="mjo_phase",
                                    dpi=180,
                                    also_pdf=True):
    os.makedirs(outdir, exist_ok=True)

    stats, vmin, vmax, stations = _prep_phase_stats(master_df, amp_min=amp_min)
    norm = colors.Normalize(vmin=vmin, vmax=vmax)
    cmap = CMAP_STAB

    for phase in PHASE_ORDER:
        info = PHASE_INFO[phase]
        lon_c = info["lon_c"]

        # layout ancho para que el mapa luzca
        fig = plt.figure(figsize=(16, 6.2))
        gs = fig.add_gridspec(1, 2, width_ratios=[3.3, 1.5], wspace=0.35)

        # MAPA
        if _HAS_CARTOPY:
            ax_map = fig.add_subplot(gs[0,0], projection=ccrs.PlateCarree())
        else:
            ax_map = fig.add_subplot(gs[0,0])
        _add_map_background(ax_map)

        LON, LAT, conv, supp = _phase_fields(lon_c)
        if _HAS_CARTOPY:
            im1 = ax_map.pcolormesh(LON, LAT, conv, transform=ccrs.PlateCarree(),
                                    shading="auto", cmap="Greens", vmin=0, vmax=1)
            im2 = ax_map.pcolormesh(LON, LAT, supp, transform=ccrs.PlateCarree(),
                                    shading="auto", cmap="BrBG_r", vmin=0, vmax=1)
        else:
            im1 = ax_map.pcolormesh(LON, LAT, conv, shading="auto",
                                    cmap="Greens", vmin=0, vmax=1)
            im2 = ax_map.pcolormesh(LON, LAT, supp, shading="auto",
                                    cmap="BrBG_r", vmin=0, vmax=1)
        im1.set_alpha(0.55); im2.set_alpha(0.45)

        # LEYENDA fuera del mapa
        fig.text(0.04, 0.90, "Envolvente MJO esquemática\nConvección (verde) · suprimida (marrón)",
                 ha="left", va="top", fontsize=13, color="white",
                 bbox=dict(facecolor="#0f0f0f", edgecolor="#4d4d4d",
                           boxstyle="round,pad=0.35"))

        # PANEL BARRAS
        ax_bar = fig.add_subplot(gs[0,1])
        _draw_stability_panel(ax_bar, stations, phase, stats, norm, cmap, vmax)

        # COLORBAR FIJA (debajo de barras)
        sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
        cbar = fig.colorbar(sm, ax=ax_bar, orientation="horizontal",
                            fraction=0.12, pad=0.22)
        cbar.set_label("N² (μ s$^{-2}$) – estabilidad", fontsize=13)
        cbar.ax.tick_params(labelsize=11)

        # TÍTULO
        fig.suptitle(
            f"Pre-acondicionamiento MJO · Fase {phase} — {info['region']}   (amp ≥ {amp_min})",
            fontsize=24, fontweight="bold", color="white", y=0.98
        )

        # GUARDAR
        png_path = os.path.join(outdir, f"{basename}_{phase:02d}.png")
        fig.savefig(png_path, bbox_inches="tight", dpi=dpi)
        plt.close(fig)
        print(f"✓ guardado: {png_path}")

    # (opcional) PDF multipágina con las 8 fases (requiere matplotlib>=3.3)
    if also_pdf:
        try:
            from PIL import Image
            pdf_path = os.path.join(outdir, f"{basename}_all_phases.pdf")
            pages = []
            for phase in PHASE_ORDER:
                with Image.open(os.path.join(outdir, f"{basename}_{phase:02d}.png")) as img:
                    pages.append(img.convert("RGB"))
            pages[0].save(pdf_path, save_all=True, append_images=pages[1:], resolution=dpi)
            for page in pages:
                page.close()
            print(f"✓ PDF multipágina: {pdf_path}")
        except Exception as e:
            print("No pude crear el PDF multipágina:", e)

# ============================
# EJECUCIÓN (ajusta amp_min si quieres)
# ============================
# Ejemplo con el DataFrame maestro diario (dos estaciones):
#   daily_2st  -> columnas: ['station','date','N2_s2','phase','amplitude', ...]
save_mjo_preconditioning_phases(
    daily_2st,
    amp_min=1.0,             # pon None para no filtrar por amplitud del MJO
    outdir="mjo_phases",     # carpeta de salida
    basename="mjo_phase",    # prefijo de archivo
    dpi=160,
    also_pdf=True
)


✓ guardado: mjo_phases\mjo_phase_01.png


✓ guardado: mjo_phases\mjo_phase_02.png


✓ guardado: mjo_phases\mjo_phase_03.png


✓ guardado: mjo_phases\mjo_phase_04.png


✓ guardado: mjo_phases\mjo_phase_05.png


✓ guardado: mjo_phases\mjo_phase_06.png


✓ guardado: mjo_phases\mjo_phase_07.png


✓ guardado: mjo_phases\mjo_phase_08.png


✓ PDF multipágina: mjo_phases\mjo_phase_all_phases.pdf


In [26]:
# ============================================================
# MJO pre-acondicionamiento · 8 IMÁGENES (una por fase)
# Versión: ANOMALÍAS N²′ respecto a climatología calendario 1981–2010
# - amp ≥ 1.0 por defecto
# - Colorbar centrada en 0 (TwoSlopeNorm)
# - Eje X simétrico y línea en 0
# ============================================================

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import colors

# ---------- (opcional) cartopy para costas ----------
_HAS_CARTOPY = False
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    _HAS_CARTOPY = True
except Exception:
    warnings.warn("cartopy no disponible: se usará un fondo simple.")

# ---------- estilo general (tema oscuro) ----------
plt.rcParams.update({
    "figure.facecolor": "black",
    "axes.facecolor":   "black",
    "savefig.facecolor":"black",
    "text.color":       "white",
    "axes.edgecolor":   "#AAAAAA",
    "axes.labelcolor":  "white",
    "xtick.color":      "white",
    "ytick.color":      "white",
    "font.size":        14,
})

# ---------- Fases → región y longitudes de centro (ºE) ----------
PHASE_INFO = {
    1: {"region": "Western Hemisphere & Africa", "lon_c": -30},
    2: {"region": "Indian Ocean (W)",            "lon_c":  60},
    3: {"region": "Indian Ocean (E)",            "lon_c":  90},
    4: {"region": "Maritime Continent (W)",      "lon_c": 120},
    5: {"region": "Maritime Continent (E)",      "lon_c": 140},
    6: {"region": "Western Pacific",             "lon_c": 160},
    7: {"region": "Central/E Pacific",           "lon_c": -140},
    8: {"region": "Africa/Atlantic (return)",    "lon_c":  -10},
}
PHASE_ORDER = [1,2,3,4,5,6,7,8]

# ---------- colormap para ANOMALÍAS (centrada en 0) ----------
CMAP_STAB = plt.cm.RdBu_r  # azul = menos estable, rojo = más estable
GRID_COL   = "#666666"

# ---------- util: preparar ANOMALÍAS N²′ por fase (μ s^-2), con misma escala ----------
def _prep_phase_stats_anom(master_df, amp_min=1.0):
    d = master_df.copy()

    # Usa exactamente N2_anom calculada antes con la climatología calendario 1981–2010.
    required = {"date", "station", "phase", "amplitude", "N2_anom"}
    missing = required.difference(d.columns)
    if missing:
        raise ValueError(f"Faltan columnas requeridas: {sorted(missing)}")
    d["date"] = pd.to_datetime(d["date"], errors="coerce")
    d = d[(d["N2_anom"].notna()) & (d["phase"].notna()) & (d["date"].notna())]

    if d.empty:
        raise ValueError("No hay datos válidos de N².")

    # Filtro por amplitud del MJO (amp ≥ amp_min)
    if amp_min is not None:
        d = d[(d["amplitude"].notna()) & (d["amplitude"] >= amp_min)]

    if d.empty:
        raise ValueError("No hay datos tras aplicar el filtro de amplitud (amp ≥ {}).".format(amp_min))

    # Media por (estación, fase) en μ s^-2
    g = (d.groupby(["station","phase"])["N2_anom"]
           .mean()
           .mul(1e6)  # a micro s^-2
           .rename("N2p_us2"))  # N²′ en μ s^-2

    # Escala simétrica centrada en 0
    max_abs = float(np.nanmax(np.abs(g.values))) if len(g) else 1.0
    if not np.isfinite(max_abs) or max_abs == 0:
        max_abs = 1.0
    vmin, vmax = -max_abs, +max_abs

    stations = sorted(d["station"].unique())
    return g, vmin, vmax, stations

# ---------- campo sintético convectivo/suprimido por fase ----------
def _phase_fields(lon_c, sigma=35.0, lat_half=18.0):
    lons = np.linspace(-180, 180, 721)
    lats = np.linspace(-30, 30, 241)
    LON, LAT = np.meshgrid(lons, lats)
    def wrapdist(a, b):
        return (a - b + 180) % 360 - 180
    gauss_lon = np.exp(-0.5*(wrapdist(LON, lon_c)/sigma)**2)
    gauss_lat = np.exp(-0.5*((LAT/lat_half))**2)
    conv = gauss_lon * gauss_lat
    supp = np.exp(-0.5*(wrapdist(LON, (lon_c+180))/sigma)**2) * gauss_lat
    return LON, LAT, conv, supp

# ---------- fondo del mapa ----------
def _add_map_background(ax):
    if _HAS_CARTOPY:
        ax.set_global()
        ax.set_extent([-180, 180, -30, 30], crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.LAND.with_scale("110m"),
                       facecolor="#2b2b2b", edgecolor="#404040", linewidth=0.4)
        ax.coastlines(color="#9A9A9A", linewidth=0.7)
        ax.gridlines(draw_labels=False, color=GRID_COL, linewidth=0.4, linestyle=":")
    else:
        ax.set_xlim(-180, 180); ax.set_ylim(-30, 30)
        for lon in range(-180, 181, 30):
            ax.plot([lon, lon], [-30, 30], color=GRID_COL, lw=0.4, ls=":")
        for lat in range(-30, 31, 10):
            ax.plot([-180, 180], [lat, lat], color=GRID_COL, lw=0.4, ls=":")
        ax.fill_between([-180, 180], -30, 30, color="#222222", alpha=0.35)
        ax.set_xlabel("Longitud (°)"); ax.set_ylabel("Latitud (°)")

# ---------- panel de barras N²′ ----------
def _draw_stability_panel(ax, stations, phase, stats_series, norm, cmap, max_abs):
    ax.clear()
    ax.set_facecolor("black")
    for sp in ax.spines.values():
        sp.set_color("#AAAAAA")

    vals = [stats_series.get((st, phase), np.nan) for st in stations]
    y = np.arange(len(stations))[::-1]

    cols = [cmap(norm(v)) if np.isfinite(v) else (0.4,0.4,0.4,0.3) for v in vals]
    ax.barh(y, [0 if not np.isfinite(v) else v for v in vals],
            color=cols, edgecolor="#E6E6E6", linewidth=1.2, height=0.55)

    # Etiquetas al final de cada barra (con signo)
    for yy, v in zip(y, vals):
        txt = "s/d" if not np.isfinite(v) else f"{v:+.0f}"
        xend = 0 if not np.isfinite(v) else v
        # desplazamiento relativo para no tapar la barra
        sign = np.sign(xend if xend != 0 else 1)
        ax.text(xend + sign*max_abs*0.04, yy, txt, va="center",
                ha=("left" if sign > 0 else "right"), fontsize=16, fontweight="bold", color="white")

    ax.set_yticks(y); ax.set_yticklabels(stations, fontsize=14)
    ax.set_xlabel("N²′ (μ s$^{-2}$)", fontsize=14, labelpad=6)

    ax.set_xlim(-max_abs*1.18, max_abs*1.18)
    ax.grid(axis="x", color="#555555", alpha=0.5, lw=0.7)
    ax.axvline(0, color="#DDDDDD", lw=1.2, alpha=0.9)

    ax.set_title(f"Fase {phase}: anomalía media", fontsize=13, pad=10)

# ---------- función principal: guarda 8 PNG (y PDF opcional) ----------
def save_mjo_preconditioning_phases_anom(master_df,
                                         amp_min=1.0,
                                         outdir="mjo_phases_anom",
                                         basename="mjo_phase_anom",
                                         dpi=180,
                                         also_pdf=True):
    os.makedirs(outdir, exist_ok=True)

    stats, vmin, vmax, stations = _prep_phase_stats_anom(master_df, amp_min=amp_min)
    # Normalización centrada en 0
    norm = colors.TwoSlopeNorm(vmin=vmin, vcenter=0.0, vmax=vmax)
    cmap = CMAP_STAB
    max_abs = max(abs(vmin), abs(vmax))

    for phase in PHASE_ORDER:
        info = PHASE_INFO[phase]
        lon_c = info["lon_c"]

        fig = plt.figure(figsize=(16, 6.2))
        gs = fig.add_gridspec(1, 2, width_ratios=[3.3, 1.5], wspace=0.35)

        # MAPA (envolvente convectiva/suprimida)
        if _HAS_CARTOPY:
            ax_map = fig.add_subplot(gs[0,0], projection=ccrs.PlateCarree())
        else:
            ax_map = fig.add_subplot(gs[0,0])
        _add_map_background(ax_map)

        LON, LAT, conv, supp = _phase_fields(lon_c)
        if _HAS_CARTOPY:
            im1 = ax_map.pcolormesh(LON, LAT, conv, transform=ccrs.PlateCarree(),
                                    shading="auto", cmap="Greens", vmin=0, vmax=1)
            im2 = ax_map.pcolormesh(LON, LAT, supp, transform=ccrs.PlateCarree(),
                                    shading="auto", cmap="BrBG_r", vmin=0, vmax=1)
        else:
            im1 = ax_map.pcolormesh(LON, LAT, conv, shading="auto",
                                    cmap="Greens", vmin=0, vmax=1)
            im2 = ax_map.pcolormesh(LON, LAT, supp, shading="auto",
                                    cmap="BrBG_r", vmin=0, vmax=1)
        im1.set_alpha(0.55); im2.set_alpha(0.45)

        # LEYENDA fuera del mapa
        fig.text(0.055, 0.84, "Envolvente MJO esquemática\nConvección (verde) · Suprimida (marrón)",
                 ha="left", va="top", fontsize=13, color="white",
                 bbox=dict(facecolor="#0f0f0f", edgecolor="#4d4d4d",
                           boxstyle="round,pad=0.35"))

        # PANEL BARRAS (N²′)
        ax_bar = fig.add_subplot(gs[0,1])
        _draw_stability_panel(ax_bar, stations, phase, stats, norm, cmap, max_abs)

        # COLORBAR centrada en 0
        sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
        cbar = fig.colorbar(sm, ax=ax_bar, orientation="horizontal",
                            fraction=0.12, pad=0.22)
        cbar.set_label("Estabilidad relativa", fontsize=12)
        cbar.ax.tick_params(labelsize=11)

        # TÍTULO
        fig.suptitle(
            f"Pre-acondicionamiento MJO · Fase {phase} — {info['region']}   (amp ≥ {amp_min})",
            fontsize=18, fontweight="bold", color="white", y=0.97
        )
        fig.subplots_adjust(left=0.06, right=0.97, top=0.84, bottom=0.14)

        # GUARDAR
        png_path = os.path.join(outdir, f"{basename}_{phase:02d}.png")
        fig.savefig(png_path, bbox_inches="tight", dpi=dpi)
        plt.close(fig)
        print(f"✓ guardado: {png_path}")

    # (opcional) PDF multipágina con las 8 fases
    if also_pdf:
        try:
            from PIL import Image
            pdf_path = os.path.join(outdir, f"{basename}_all_phases.pdf")
            pages = []
            for phase in PHASE_ORDER:
                with Image.open(os.path.join(outdir, f"{basename}_{phase:02d}.png")) as img:
                    pages.append(img.convert("RGB"))
            pages[0].save(pdf_path, save_all=True, append_images=pages[1:], resolution=dpi)
            for page in pages:
                page.close()
            print(f"✓ PDF multipágina: {pdf_path}")
        except Exception as e:
            print("No pude crear el PDF multipágina:", e)

# ============================
# EJECUCIÓN
# ============================
# metrics_anom contiene N2_anom basada en climatología calendario 1981–2010.
save_mjo_preconditioning_phases_anom(
    metrics_anom,
    amp_min=1.0,                 # filtro de amplitud (MJO activo)
    outdir="mjo_phases_anom",    # carpeta de salida
    basename="mjo_phase_anom",   # prefijo de archivo
    dpi=160,
    also_pdf=True
)


✓ guardado: mjo_phases_anom\mjo_phase_anom_01.png


✓ guardado: mjo_phases_anom\mjo_phase_anom_02.png


✓ guardado: mjo_phases_anom\mjo_phase_anom_03.png


✓ guardado: mjo_phases_anom\mjo_phase_anom_04.png


✓ guardado: mjo_phases_anom\mjo_phase_anom_05.png


✓ guardado: mjo_phases_anom\mjo_phase_anom_06.png


✓ guardado: mjo_phases_anom\mjo_phase_anom_07.png


✓ guardado: mjo_phases_anom\mjo_phase_anom_08.png


✓ PDF multipágina: mjo_phases_anom\mjo_phase_anom_all_phases.pdf
